In [ ]:
import sys
print(sys.executable)
print(sys.version)

In [ ]:
import duckdb
import pandas as pd
from pathlib import Path

print("DuckDB version:", duckdb.__version__)
print("Pandas version:", pd.__version__)
print("Current working directory:", Path.cwd())

In [ ]:
from pathlib import Path
import duckdb

project_root = Path.cwd()
db_path = project_root / "05_Starter" / "pulseiq.duckdb"

print("Database exists:", db_path.exists())
print("Database path:", db_path)

con = duckdb.connect(str(db_path), read_only=True)

tables = con.execute("""
    SHOW TABLES FROM raw
""").fetchdf()

print("\nNumber of raw tables:", len(tables))
display(tables)

In [ ]:
contracts = con.execute("""
    SELECT *
    FROM raw.data_contracts
    ORDER BY feed, column_name
""").fetchdf()

print("Total contract rules:", len(contracts))
print("\nColumns in the contract table:")
print(contracts.columns.tolist())

display(contracts.head(15))

In [ ]:
feed_summary = con.execute("""
    SELECT
        feed,
        COUNT(*) AS contract_columns,
        SUM(CASE WHEN nullable = FALSE THEN 1 ELSE 0 END)
            AS required_columns,
        SUM(CASE WHEN nullable = TRUE THEN 1 ELSE 0 END)
            AS nullable_columns
    FROM raw.data_contracts
    GROUP BY feed
    ORDER BY feed
""").fetchdf()

display(feed_summary)

In [ ]:
contract_feeds = con.execute("""
    SELECT DISTINCT feed
    FROM raw.data_contracts
    ORDER BY feed
""").fetchdf()

print("Feeds covered by contracts:")
display(contract_feeds)

print("\nOur three initial tables:")
print("- fact_payroll")
print("- fact_attendance_monthly")
print("- dim_employee")

In [ ]:
dq_catalog = con.execute("""
    SELECT *
    FROM raw.dq_checks_catalog
""").fetchdf()

print("Total data-quality checks:", len(dq_catalog))
print("Available columns:", dq_catalog.columns.tolist())

display(dq_catalog)

In [ ]:
table_names = con.execute("""
    SHOW TABLES FROM raw
""").fetchdf()["name"].tolist()

print("Known-issues table loaded:", "dq_known_issues" in table_names)
print("\nTables containing 'issue' or 'dq':")
print([
    name for name in table_names
    if "issue" in name.lower() or "dq" in name.lower()
])

In [ ]:
import pandas as pd
from pathlib import Path

workbook_path = Path.cwd() / "03_Data" / "PulseIQ_Dataset.xlsx"

print("Workbook exists:", workbook_path.exists())

excel = pd.ExcelFile(workbook_path)
print("\nKnown-issues sheet exists:", "dq_known_issues" in excel.sheet_names)

if "dq_known_issues" in excel.sheet_names:
    known_issues = pd.read_excel(
        workbook_path,
        sheet_name="dq_known_issues"
    )

    print("\nShape:", known_issues.shape)
    print("Columns:", known_issues.columns.tolist())
    display(known_issues)

In [ ]:
for i, sheet in enumerate(excel.sheet_names, start=1):
    print(f"{i}. {sheet}")

In [ ]:
readme = pd.read_excel(
    workbook_path,
    sheet_name="README",
    header=None
)

print("README shape:", readme.shape)
display(readme)

In [ ]:
for i, row in readme.iterrows():
    values = [
        str(value).strip()
        for value in row.tolist()
        if pd.notna(value) and str(value).strip()
    ]

    if values:
        print(f"Row {i}: " + " | ".join(values))

In [ ]:
contract_feed_tables = {
    "AI anomaly results (fact_anomaly)": "fact_anomaly",
    "AI attrition risk (fact_attrition_risk)": "fact_attrition_risk",
    "AI forecast (fact_forecast)": "fact_forecast",
    "Task board (fact_task)": "fact_task",
}

for feed, table in contract_feed_tables.items():
    contract_columns = con.execute(
        """
        SELECT column_name, data_type, nullable,
               rule_or_allowed_values
        FROM raw.data_contracts
        WHERE feed = ?
        ORDER BY column_name
        """,
        [feed]
    ).fetchdf()

    actual_columns = con.execute(
        f"DESCRIBE raw.{table}"
    ).fetchdf()["column_name"].tolist()

    expected_columns = contract_columns["column_name"].tolist()

    print(f"\n{'=' * 60}")
    print(f"Feed: {feed}")
    print(f"Table: raw.{table}")
    print("Contract columns:", len(expected_columns))
    print("Actual table columns:", len(actual_columns))
    print("Missing contract columns:",
          sorted(set(expected_columns) - set(actual_columns)))
    print("Undocumented actual columns:",
          sorted(set(actual_columns) - set(expected_columns)))

In [ ]:
feed = "AI anomaly results (fact_anomaly)"
table = "fact_anomaly"

contract = con.execute("""
    SELECT column_name, data_type, nullable,
           rule_or_allowed_values
    FROM raw.data_contracts
    WHERE feed = ?
    ORDER BY column_name
""", [feed]).fetchdf()

actual = con.execute("""
    DESCRIBE raw.fact_anomaly
""").fetchdf()

null_counts = con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) - COUNT(result_id) AS missing_result_id,
        COUNT(*) - COUNT(employee_id) AS missing_employee_id,
        COUNT(*) - COUNT(anomaly_score) AS missing_anomaly_score,
        COUNT(*) - COUNT(flagged) AS missing_flagged
    FROM raw.fact_anomaly
""").fetchdf()

print("Contract:")
display(contract)

print("Actual schema:")
display(actual)

print("Required-field null check:")
display(null_counts)

In [ ]:
import pandas as pd

feed = "AI anomaly results (fact_anomaly)"
table = "fact_anomaly"

contract = con.execute("""
    SELECT column_name, nullable
    FROM raw.data_contracts
    WHERE feed = ?
""", [feed]).fetchdf()

actual_columns = set(
    con.execute(f"DESCRIBE raw.{table}")
    .fetchdf()["column_name"]
)

required_columns = contract.loc[
    contract["nullable"] == False, "column_name"
].tolist()

missing_columns = [
    col for col in required_columns
    if col not in actual_columns
]

if missing_columns:
    print("Required contract columns missing from table:", missing_columns)
else:
    expressions = ",\n".join(
        f'COUNT(*) - COUNT("{col}") AS "{col}"'
        for col in required_columns
    )

    null_summary = con.execute(
        f"SELECT {expressions} FROM raw.{table}"
    ).fetchdf()

    print("Required columns checked:", len(required_columns))
    display(null_summary.T.rename(columns={0: "missing_values"}))

In [ ]:
anomaly_checks = con.execute("""
    SELECT
        COUNT(*) AS total_rows,

        COUNT(DISTINCT result_id) AS unique_result_ids,

        COUNT(*) FILTER (
            WHERE NOT regexp_full_match(result_id, 'AN-[0-9]{4}')
        ) AS invalid_result_id_format,

        COUNT(*) FILTER (
            WHERE anomaly_score < 0 OR anomaly_score > 1
        ) AS invalid_anomaly_scores,

        COUNT(*) FILTER (
            WHERE flagged NOT IN ('Y', 'N')
        ) AS invalid_flagged_values,

        COUNT(*) FILTER (
            WHERE employee_id NOT IN (
                SELECT employee_id
                FROM raw.dim_employee
            )
        ) AS unmatched_employee_ids

    FROM raw.fact_anomaly
""").fetchdf()

display(anomaly_checks)

In [ ]:
print("Total rows:")
display(con.execute("""
    SELECT COUNT(*) AS total_rows
    FROM raw.fact_anomaly
""").fetchdf())

print("ID uniqueness:")
display(con.execute("""
    SELECT
        COUNT(result_id) AS non_null_ids,
        COUNT(DISTINCT result_id) AS unique_ids,
        COUNT(*) - COUNT(DISTINCT result_id) AS duplicate_excess
    FROM raw.fact_anomaly
""").fetchdf())

print("Sample result IDs:")
display(con.execute("""
    SELECT result_id
    FROM raw.fact_anomaly
    LIMIT 10
""").fetchdf())

In [ ]:
anomaly_checks = con.execute("""
    SELECT
        (SELECT COUNT(*)
         FROM raw.fact_anomaly) AS total_rows,

        (SELECT COUNT(*)
         FROM raw.fact_anomaly
         WHERE NOT regexp_full_match(result_id, 'AN-[0-9]{4}')
        ) AS invalid_result_id_format,

        (SELECT COUNT(*)
         FROM raw.fact_anomaly
         WHERE anomaly_score IS NULL
            OR anomaly_score < 0
            OR anomaly_score > 1
        ) AS invalid_anomaly_scores,

        (SELECT COUNT(*)
         FROM raw.fact_anomaly
         WHERE flagged IS NULL
            OR flagged NOT IN ('Y', 'N')
        ) AS invalid_flagged_values,

        (SELECT COUNT(*)
         FROM raw.fact_anomaly a
         WHERE NOT EXISTS (
             SELECT 1
             FROM raw.dim_employee e
             WHERE e.employee_id = a.employee_id
         )
        ) AS unmatched_employee_ids
""").fetchdf()

display(anomaly_checks)
print("1. Row count:")
print(con.execute(
    "SELECT COUNT(*) FROM raw.fact_anomaly"
).fetchone())

print("\n2. First 5 IDs:")
print(con.execute("""
    SELECT result_id, LENGTH(result_id),
           regexp_full_match(result_id, 'AN-[0-9]{4}') AS format_ok
    FROM raw.fact_anomaly
    LIMIT 5
""").fetchdf().to_string(index=False))

print("\n3. DataFrame result:")
print(anomaly_checks.to_string(index=False))

In [ ]:
display(con.execute("""
    SELECT
        COUNT(*) AS total_rows,

        COUNT(*) FILTER (
            WHERE money_at_risk IS NOT NULL
              AND money_at_risk < 0
        ) AS negative_money_at_risk,

        COUNT(DISTINCT reason_code) AS distinct_reason_codes,

        COUNT(*) FILTER (
            WHERE reason_code IS NULL
               OR TRIM(reason_code) = ''
        ) AS missing_reason_codes,

        COUNT(DISTINCT source_dataset) AS distinct_source_datasets,

        COUNT(*) FILTER (
            WHERE source_dataset IS NULL
               OR TRIM(source_dataset) = ''
        ) AS missing_source_datasets,

        COUNT(*) FILTER (
            WHERE scored_at IS NULL
        ) AS missing_scored_at

    FROM raw.fact_anomaly
""").fetchdf())
result = con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT reason_code) AS distinct_reason_codes,
        COUNT(DISTINCT source_dataset) AS distinct_source_datasets,
        COUNT(*) FILTER (
            WHERE money_at_risk < 0
        ) AS negative_money_at_risk,
        COUNT(*) FILTER (
            WHERE reason_code IS NULL
               OR TRIM(reason_code) = ''
        ) AS missing_reason_codes
    FROM raw.fact_anomaly
""").fetchdf()

print(result.to_string(index=False))
print("\nRaw tuple:", con.execute("""
    SELECT COUNT(*), COUNT(DISTINCT reason_code)
    FROM raw.fact_anomaly
""").fetchone())

In [ ]:
print("Reason codes:")
display(con.execute("""
    SELECT reason_code, COUNT(*) AS record_count
    FROM raw.fact_anomaly
    GROUP BY reason_code
    ORDER BY reason_code
""").fetchdf().to_string(index=False))

print("\nSource datasets:")
display(con.execute("""
    SELECT source_dataset, COUNT(*) AS record_count
    FROM raw.fact_anomaly
    GROUP BY source_dataset
    ORDER BY source_dataset
""").fetchdf().to_string(index=False))

In [ ]:
print("Official source_dataset contract rule:")

display(con.execute("""
    SELECT
        column_name,
        rule_or_allowed_values,
        description
    FROM raw.data_contracts
    WHERE feed = 'AI anomaly results (fact_anomaly)'
      AND column_name = 'source_dataset'
""").fetchdf().to_string(index=False))

print("\nObserved source_dataset values:")

display(con.execute("""
    SELECT DISTINCT source_dataset
    FROM raw.fact_anomaly
    ORDER BY source_dataset
""").fetchdf().to_string(index=False))

In [ ]:
print("Flagged values:")
print(con.execute("""
    SELECT flagged, COUNT(*) AS record_count
    FROM raw.fact_anomaly
    GROUP BY flagged
    ORDER BY flagged
""").fetchdf().to_string(index=False))

print("\nAnomaly score range:")
print(con.execute("""
    SELECT
        MIN(anomaly_score) AS minimum_score,
        MAX(anomaly_score) AS maximum_score,
        COUNT(*) FILTER (
            WHERE anomaly_score < 0 OR anomaly_score > 1
        ) AS out_of_range_scores,
        COUNT(*) FILTER (
            WHERE anomaly_score IS NULL
        ) AS missing_scores
    FROM raw.fact_anomaly
""").fetchdf().to_string(index=False))

In [ ]:
display(con.execute("""
    SELECT
        source_dataset,
        COUNT(*) AS total_records,
        COUNT(*) FILTER (
            WHERE record_id IS NULL OR TRIM(record_id) = ''
        ) AS missing_record_ids
    FROM raw.fact_anomaly
    GROUP BY source_dataset
    ORDER BY source_dataset
""").fetchdf().to_string(index=False))

In [ ]:
display(con.execute("""
    SELECT
        a.source_dataset,
        COUNT(*) AS total_records,
        COUNT(*) FILTER (
            WHERE a.record_id IS NOT NULL
              AND (
                  (a.source_dataset = 'payroll_monthly_features'
                   AND NOT EXISTS (
                       SELECT 1
                       FROM raw.fact_payroll p
                       WHERE p.payroll_item_id = a.record_id
                  ))
                  OR
                  (a.source_dataset = 'attendance_monthly_features'
                   AND NOT EXISTS (
                       SELECT 1
                       FROM raw.fact_attendance_monthly t
                       WHERE t.attendance_fact_id = a.record_id
                  ))
              )
        ) AS unmatched_record_ids
    FROM raw.fact_anomaly a
    GROUP BY a.source_dataset
    ORDER BY a.source_dataset
""").fetchdf().to_string(index=False))

In [ ]:
print("Anomaly record IDs for attendance:")
display(con.execute("""
    SELECT record_id
    FROM raw.fact_anomaly
    WHERE source_dataset = 'attendance_monthly_features'
    LIMIT 10
""").fetchdf())

print("\nAttendance table columns:")
display(con.execute("""
    DESCRIBE raw.fact_attendance_monthly
""").fetchdf())

print("\nSample attendance IDs:")
display(con.execute("""
    SELECT *
    FROM raw.fact_attendance_monthly
    LIMIT 5
""").fetchdf())

In [ ]:
print("Attendance anomaly record IDs:")
display(con.execute("""
    SELECT record_id, employee_id, period
    FROM raw.fact_anomaly
    WHERE source_dataset = 'attendance_monthly_features'
    LIMIT 10
""").fetchdf())

print("\nDo AF IDs appear anywhere in attendance_fact_id?")
display(con.execute("""
    SELECT COUNT(*) AS matching_ids
    FROM raw.fact_anomaly a
    JOIN raw.fact_attendance_monthly t
      ON a.record_id = t.attendance_fact_id
    WHERE a.source_dataset = 'attendance_monthly_features'
""").fetchdf())

In [ ]:
display(con.execute("""
    SELECT sheet, "column", data_type, description, example
    FROM raw.data_dictionary
    WHERE
        (sheet = 'fact_attendance_monthly'
         AND LOWER("column") LIKE '%id%')
        OR
        (sheet = 'fact_anomaly'
         AND LOWER("column") LIKE '%record_id%')
    ORDER BY sheet, "column"
""").fetchdf().to_string(index=False))

In [ ]:
display(con.execute("""
    SELECT
        COUNT(*) AS anomaly_records,
        COUNT(*) FILTER (
            WHERE EXISTS (
                SELECT 1
                FROM raw.fact_attendance_monthly t
                WHERE t.employee_id = a.employee_id
                  AND date_trunc('month', t.period_end_date)
                      = date_trunc('month', a.period)
            )
        ) AS employee_month_matches,
        COUNT(*) FILTER (
            WHERE NOT EXISTS (
                SELECT 1
                FROM raw.fact_attendance_monthly t
                WHERE t.employee_id = a.employee_id
                  AND date_trunc('month', t.period_end_date)
                      = date_trunc('month', a.period)
            )
        ) AS no_employee_month_match
    FROM raw.fact_anomaly a
    WHERE a.source_dataset = 'attendance_monthly_features'
""").fetchdf().to_string(index=False))

In [ ]:
display(con.execute("""
    SELECT
        COUNT(*) AS anomaly_records,
        COUNT(*) FILTER (
            WHERE match_count = 1
        ) AS unique_employee_month_matches,
        COUNT(*) FILTER (
            WHERE match_count > 1
        ) AS ambiguous_employee_month_matches,
        COUNT(*) FILTER (
            WHERE match_count = 0
        ) AS missing_employee_month_matches
    FROM (
        SELECT
            a.result_id,
            (
                SELECT COUNT(*)
                FROM raw.fact_attendance_monthly t
                WHERE t.employee_id = a.employee_id
                  AND date_trunc('month', t.period_end_date)
                      = date_trunc('month', a.period)
            ) AS match_count
        FROM raw.fact_anomaly a
        WHERE a.source_dataset = 'attendance_monthly_features'
    ) x
""").fetchdf().to_string(index=False))

In [ ]:
print(con.execute("""
    SELECT
        source_dataset,
        COUNT(*) AS total_records,
        COUNT(*) FILTER (
            WHERE money_at_risk IS NULL
        ) AS null_money_at_risk,
        COUNT(*) FILTER (
            WHERE money_at_risk IS NOT NULL
        ) AS populated_money_at_risk
    FROM raw.fact_anomaly
    GROUP BY source_dataset
    ORDER BY source_dataset
""").fetchdf().to_string(index=False))

In [ ]:
print(con.execute("""
    SELECT
        COUNT(*) AS payroll_records,
        COUNT(*) FILTER (
            WHERE money_at_risk IS NULL
        ) AS missing_money_values,
        COUNT(*) FILTER (
            WHERE money_at_risk < 0
        ) AS negative_money_values,
        MIN(money_at_risk) AS minimum_money_at_risk,
        MAX(money_at_risk) AS maximum_money_at_risk
    FROM raw.fact_anomaly
    WHERE source_dataset = 'payroll_monthly_features'
""").fetchdf().to_string(index=False))

In [ ]:
print(con.execute("""
    SELECT
        model_version,
        COUNT(*) AS record_count
    FROM raw.fact_anomaly
    GROUP BY model_version
    ORDER BY model_version
""").fetchdf().to_string(index=False))

In [ ]:
print(con.execute("""
    SELECT
        COUNT(*) AS total_records,
        COUNT(*) FILTER (
            WHERE reason_code IS NULL OR TRIM(reason_code) = ''
        ) AS missing_reason_codes,
        COUNT(*) FILTER (
            WHERE reason_text IS NULL OR TRIM(reason_text) = ''
        ) AS missing_reason_text,
        COUNT(DISTINCT reason_code) AS distinct_reason_codes
    FROM raw.fact_anomaly
""").fetchdf().to_string(index=False))

In [ ]:
print(con.execute("""
    SELECT
        COUNT(*) AS total_records,
        COUNT(*) FILTER (
            WHERE scored_at IS NULL
        ) AS missing_timestamps,
        MIN(scored_at) AS earliest_timestamp,
        MAX(scored_at) AS latest_timestamp,
        COUNT(DISTINCT CAST(scored_at AS DATE)) AS distinct_timestamp_dates
    FROM raw.fact_anomaly
""").fetchdf().to_string(index=False))

In [ ]:
print(con.execute("""
    SELECT
        scored_at,
        COUNT(*) AS record_count
    FROM raw.fact_anomaly
    GROUP BY scored_at
    ORDER BY scored_at
    LIMIT 10
""").fetchdf().to_string(index=False))

In [ ]:
anomaly_validation_summary = {
    "feed": "fact_anomaly",
    "total_rows": 405,
    "unique_result_ids": 405,
    "invalid_result_id_format": 0,
    "invalid_anomaly_scores": 0,
    "invalid_flagged_values": 0,
    "missing_required_fields": 0,
    "distinct_reason_codes": 11,
    "distinct_source_datasets": 2,
    "missing_reason_text": 0,
    "missing_scored_at": 0,
    "attendance_employee_month_matches": 221,
    "attendance_record_id_mapping": "unresolved",
    "timestamp_timezone": "not confirmed"
}

for key, value in anomaly_validation_summary.items():
    print(f"{key}: {value}")

In [ ]:
attrition_contract = con.execute("""
    SELECT
        column_name,
        data_type,
        nullable,
        rule_or_allowed_values,
        description
    FROM raw.data_contracts
    WHERE feed = 'AI attrition risk (fact_attrition_risk)'
    ORDER BY column_name
""").fetchdf()

print("Contract columns:", len(attrition_contract))
display(attrition_contract)

In [ ]:
display(con.execute("""
    SELECT
        COUNT(*) AS total_rows,

        COUNT(*) FILTER (
            WHERE prediction_id IS NULL
        ) AS missing_prediction_id,

        COUNT(*) FILTER (
            WHERE as_of_date IS NULL
        ) AS missing_as_of_date,

        COUNT(*) FILTER (
            WHERE department_id IS NULL
        ) AS missing_department_id,

        COUNT(*) FILTER (
            WHERE employee_id IS NULL
        ) AS missing_employee_id,

        COUNT(*) FILTER (
            WHERE model_version IS NULL
        ) AS missing_model_version,

        COUNT(*) FILTER (
            WHERE risk_band IS NULL
        ) AS missing_risk_band,

        COUNT(*) FILTER (
            WHERE risk_score_12m IS NULL
        ) AS missing_risk_score

    FROM raw.fact_attrition_risk
""").fetchdf().to_string(index=False))

In [ ]:
display(con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT prediction_id) AS unique_prediction_ids,
        COUNT(*) FILTER (
            WHERE NOT regexp_matches(prediction_id, '^AR-[0-9]{4}$')
        ) AS invalid_prediction_id_format,
        COUNT(*) FILTER (
            WHERE risk_score_12m < 0 OR risk_score_12m > 1
        ) AS invalid_risk_scores,
        MIN(risk_score_12m) AS minimum_risk_score,
        MAX(risk_score_12m) AS maximum_risk_score
    FROM raw.fact_attrition_risk
""").fetchdf().to_string(index=False))

In [ ]:
display(con.execute("""
    SELECT
        risk_band,
        COUNT(*) AS record_count,
        MIN(risk_score_12m) AS minimum_score,
        MAX(risk_score_12m) AS maximum_score
    FROM raw.fact_attrition_risk
    GROUP BY risk_band
    ORDER BY risk_band
""").fetchdf().to_string(index=False))

In [ ]:
display(con.execute("""
    SELECT
        COUNT(*) AS total_predictions,

        COUNT(*) FILTER (
            WHERE d.department_id IS NULL
        ) AS missing_department_matches,

        COUNT(*) FILTER (
            WHERE e.employee_id IS NULL
        ) AS missing_employee_matches

    FROM raw.fact_attrition_risk AS a

    LEFT JOIN raw.dim_department AS d
        ON a.department_id = d.department_id

    LEFT JOIN (
        SELECT DISTINCT employee_id
        FROM raw.dim_employee
    ) AS e
        ON a.employee_id = e.employee_id
""").fetchdf().to_string(index=False))

In [ ]:
display(con.execute("""
    DESCRIBE raw.dim_employee
""").fetchdf().to_string(index=False))

In [ ]:
display(con.execute("""
    SELECT
        COUNT(*) AS total_predictions,

        COUNT(*) FILTER (
            WHERE e.employee_key IS NULL
        ) AS no_employee_version_on_as_of_date,

        COUNT(*) FILTER (
            WHERE e.employee_key IS NOT NULL
              AND (
                  e.hire_date > a.as_of_date
                  OR (
                      e.exit_date IS NOT NULL
                      AND e.exit_date < a.as_of_date
                  )
                  OR LOWER(TRIM(e.employment_status)) <> 'active'
              )
        ) AS predictions_for_inactive_employees,

        COUNT(*) FILTER (
            WHERE e.employee_key IS NOT NULL
              AND e.hire_date <= a.as_of_date
              AND (
                  e.exit_date IS NULL
                  OR e.exit_date >= a.as_of_date
              )
              AND LOWER(TRIM(e.employment_status)) = 'active'
        ) AS active_employee_predictions

    FROM raw.fact_attrition_risk AS a

    LEFT JOIN raw.dim_employee AS e
        ON a.employee_id = e.employee_id
       AND a.as_of_date >= e.valid_from
       AND (e.valid_to IS NULL OR a.as_of_date < e.valid_to)
""").fetchdf().to_string(index=False))

In [ ]:
display(con.execute("""
    SELECT
        COUNT(*) AS total_rows,

        COUNT(*) FILTER (
            WHERE top_reason_1 IS NULL
               OR TRIM(top_reason_1) = ''
        ) AS missing_reason_1,

        COUNT(*) FILTER (
            WHERE top_reason_2 IS NULL
               OR TRIM(top_reason_2) = ''
        ) AS missing_reason_2,

        COUNT(*) FILTER (
            WHERE top_reason_3 IS NULL
               OR TRIM(top_reason_3) = ''
        ) AS missing_reason_3

    FROM raw.fact_attrition_risk
""").fetchdf().to_string(index=False))

In [ ]:
display(con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT as_of_date) AS distinct_prediction_dates,
        MIN(as_of_date) AS earliest_prediction_date,
        MAX(as_of_date) AS latest_prediction_date,
        COUNT(DISTINCT model_version) AS distinct_model_versions,
        COUNT(*) FILTER (
            WHERE TRIM(model_version) = ''
        ) AS blank_model_versions
    FROM raw.fact_attrition_risk
""").fetchdf().to_string(index=False))

display(con.execute("""
    SELECT
        model_version,
        COUNT(*) AS record_count
    FROM raw.fact_attrition_risk
    GROUP BY model_version
    ORDER BY model_version
""").fetchdf().to_string(index=False))

In [ ]:
display(con.execute("""
    SELECT
        as_of_date,
        COUNT(*) AS record_count,
        COUNT(DISTINCT department_id) AS departments,
        COUNT(DISTINCT employee_id) AS employees
    FROM raw.fact_attrition_risk
    GROUP BY as_of_date
    ORDER BY as_of_date
""").fetchdf().to_string(index=False))

In [ ]:
display(con.execute("""
    SELECT
        prediction_id,
        as_of_date,
        employee_id,
        department_id,
        risk_band,
        risk_score_12m,
        model_version,
        model_note
    FROM raw.fact_attrition_risk
    WHERE as_of_date <> (
        SELECT MAX(as_of_date)
        FROM raw.fact_attrition_risk
    )
    ORDER BY prediction_id
""").fetchdf().to_string(index=False))

In [ ]:
display(con.execute("""
    SELECT
        as_of_date,
        risk_band,
        COUNT(*) AS record_count,
        MIN(risk_score_12m) AS minimum_score,
        MAX(risk_score_12m) AS maximum_score
    FROM raw.fact_attrition_risk
    GROUP BY as_of_date, risk_band
    ORDER BY as_of_date, risk_band
""").fetchdf().to_string(index=False))

In [ ]:
display(con.execute("""
    WITH ranked AS (
        SELECT
            prediction_id,
            as_of_date,
            employee_id,
            risk_band,
            risk_score_12m,
            NTILE(10) OVER (
                PARTITION BY as_of_date
                ORDER BY risk_score_12m DESC
            ) AS score_decile
        FROM raw.fact_attrition_risk
    )
    SELECT
        as_of_date,
        risk_band,
        MIN(risk_score_12m) AS minimum_score,
        MAX(risk_score_12m) AS maximum_score,
        MIN(score_decile) AS best_decile,
        MAX(score_decile) AS worst_decile,
        COUNT(*) AS record_count
    FROM ranked
    GROUP BY as_of_date, risk_band
    ORDER BY as_of_date, risk_band
""").fetchdf().to_string(index=False))

In [ ]:
display(con.execute("""
    SELECT
        risk_band,
        risk_score_12m,
        COUNT(*) AS record_count
    FROM raw.fact_attrition_risk
    WHERE as_of_date = DATE '2026-09-30'
      AND risk_score_12m BETWEEN 0.120 AND 0.135
    GROUP BY risk_band, risk_score_12m
    ORDER BY risk_score_12m DESC, risk_band
""").fetchdf().to_string(index=False))

In [ ]:
display(con.execute("""
    SELECT
        risk_score_12m,
        COUNT(*) AS total_records,
        COUNT(DISTINCT risk_band) AS distinct_bands,
        STRING_AGG(DISTINCT risk_band, ', ' ORDER BY risk_band) AS bands
    FROM raw.fact_attrition_risk
    WHERE as_of_date = DATE '2026-09-30'
    GROUP BY risk_score_12m
    HAVING COUNT(DISTINCT risk_band) > 1
    ORDER BY risk_score_12m DESC
""").fetchdf().to_string(index=False))

In [ ]:
attrition_validation_summary = {
    "feed": "fact_attrition_risk",
    "total_rows": 430,
    "missing_required_fields": 0,
    "duplicate_prediction_ids": 0,
    "invalid_prediction_id_format": 0,
    "invalid_risk_scores": 0,
    "invalid_risk_band_values": 0,
    "missing_department_references": 0,
    "missing_employee_references": 0,
    "missing_employee_version_on_as_of_date": 0,
    "predictions_for_inactive_employees": 0,
    "missing_top_reason_1": 0,
    "missing_top_reason_2": 0,
    "missing_top_reason_3": 0,
    "distinct_as_of_dates": 2,
    "contract_issue": "as_of_date is not constant across the batch",
    "mixed_band_for_score_0.127": True,
    "mixed_band_records_at_score_0.127": 13,
    "percentile_band_assignment": "requires further investigation",
    "raw_data_modified": False
}

for check, result in attrition_validation_summary.items():
    print(f"{check}: {result}")

In [ ]:
from pathlib import Path

notes_path = Path("docs/PQ-01_contract_notes.md")

attrition_notes = """

## fact_attrition_risk — Validation evidence

- Total input records: 430
- Missing required fields: 0
- Duplicate prediction IDs: 0
- Invalid prediction ID formats: 0
- Risk scores outside 0–1: 0
- Invalid risk-band values: 0
- Missing department references: 0
- Missing employee references: 0
- Missing employee-history version on prediction date: 0
- Predictions for inactive employees: 0
- Missing optional reason fields: 0
- Distinct prediction dates: 2
- Contract finding: as_of_date must be constant across a batch,
  but five records use 2026-03-31 and 425 use 2026-09-30.
- Risk-band finding: score 0.127 occurs in both Low and Medium
  bands across 13 records; percentile assignment requires investigation.
- Model version: attrition-logreg-baseline-0.1 (430 records)
- Raw source data modified: No

Status: Initial validation checks completed. The date contract
violation and percentile-band assignment remain unresolved.
These findings do not authorize deleting or quarantining records.
"""

if not notes_path.exists():
    raise FileNotFoundError(
        f"Notes file not found: {notes_path.resolve()}"
    )

existing_notes = notes_path.read_text(encoding="utf-8")

if "## fact_attrition_risk — Validation evidence" in existing_notes:
    print("Attrition evidence section already exists; no duplicate added.")
else:
    with notes_path.open("a", encoding="utf-8") as f:
        f.write(attrition_notes)

    print("Attrition evidence saved:", notes_path.resolve())

print("File exists:", notes_path.exists())

In [ ]:
display(con.execute("""
    SELECT *
    FROM raw.data_contracts
    WHERE LOWER(CAST(table_name AS VARCHAR)) LIKE '%forecast%'
""").fetchdf().to_string(index=False))

In [ ]:
display(con.execute("""
    DESCRIBE raw.data_contracts
""").fetchdf().to_string(index=False))

In [ ]:
display(con.execute("""
    SELECT
        feed,
        column_name,
        data_type,
        nullable,
        rule_or_allowed_values,
        description
    FROM raw.data_contracts
    WHERE LOWER(feed) LIKE '%forecast%'
    ORDER BY column_name
""").fetchdf().to_string(index=False))

In [ ]:
print("FORECAST TABLE STRUCTURE")

print(con.execute("""
    DESCRIBE raw.fact_forecast
""").fetchdf().to_string(index=False))

print("\nTOTAL FORECAST RECORDS")

print(con.execute("""
    SELECT COUNT(*) AS total_rows
    FROM raw.fact_forecast
""").fetchdf().to_string(index=False))

In [ ]:
display(con.execute("""
    SELECT
        COUNT(*) AS total_rows,

        COUNT(*) FILTER (
            WHERE forecast_id IS NULL OR TRIM(forecast_id) = ''
        ) AS missing_forecast_id,

        COUNT(*) FILTER (
            WHERE metric IS NULL OR TRIM(metric) = ''
        ) AS missing_metric,

        COUNT(*) FILTER (
            WHERE department_id IS NULL OR TRIM(department_id) = ''
        ) AS missing_department_id,

        COUNT(*) FILTER (
            WHERE forecast_month IS NULL
        ) AS missing_forecast_month,

        COUNT(*) FILTER (
            WHERE forecast_value IS NULL
        ) AS missing_forecast_value,

        COUNT(*) FILTER (
            WHERE lower_80 IS NULL
        ) AS missing_lower_80,

        COUNT(*) FILTER (
            WHERE upper_80 IS NULL
        ) AS missing_upper_80,

        COUNT(*) FILTER (
            WHERE method IS NULL OR TRIM(method) = ''
        ) AS missing_method,

        COUNT(*) FILTER (
            WHERE trained_through IS NULL
               OR TRIM(trained_through) = ''
        ) AS missing_trained_through

    FROM raw.fact_forecast
""").fetchdf().to_string(index=False))

In [ ]:
display(con.execute("""
    SELECT
        COUNT(*) AS total_rows,

        COUNT(DISTINCT forecast_id) AS unique_forecast_ids,

        COUNT(*) FILTER (
            WHERE NOT regexp_matches(forecast_id, '^FC-[0-9]{4}$')
        ) AS invalid_forecast_id_format,

        COUNT(*) FILTER (
            WHERE metric NOT IN (
                'headcount_end',
                'attrition_rate_monthly_pct'
            )
        ) AS invalid_metric_values,

        COUNT(*) FILTER (
            WHERE department_id <> 'ALL'
              AND department_id NOT IN (
                  SELECT department_id
                  FROM raw.dim_department
              )
        ) AS invalid_department_references,

        COUNT(*) FILTER (
            WHERE forecast_month <> DATE_TRUNC(
                'month', forecast_month
            )::DATE
        ) AS invalid_forecast_months

    FROM raw.fact_forecast
""").fetchdf().to_string(index=False))

In [ ]:
display(con.execute("""
    SELECT
        COUNT(*) AS total_rows,

        COUNT(*) FILTER (
            WHERE forecast_value < 0
        ) AS negative_forecast_values,

        COUNT(*) FILTER (
            WHERE lower_80 > forecast_value
        ) AS invalid_lower_bounds,

        COUNT(*) FILTER (
            WHERE upper_80 < forecast_value
        ) AS invalid_upper_bounds,

        COUNT(*) FILTER (
            WHERE NOT regexp_matches(
                trained_through, '^[0-9]{4}-(0[1-9]|1[0-2])$'
            )
        ) AS invalid_trained_through_format,

        MIN(forecast_value) AS minimum_forecast_value,
        MAX(forecast_value) AS maximum_forecast_value

    FROM raw.fact_forecast
""").fetchdf().to_string(index=False))

In [ ]:
display(con.execute("""
    SELECT
        metric,
        department_id,
        COUNT(*) AS record_count,
        MIN(forecast_month) AS first_forecast_month,
        MAX(forecast_month) AS last_forecast_month,
        MIN(forecast_value) AS minimum_forecast,
        MAX(forecast_value) AS maximum_forecast,
        MIN(lower_80) AS minimum_lower_bound,
        MAX(upper_80) AS maximum_upper_bound,
        COUNT(DISTINCT trained_through) AS training_cutoff_count
    FROM raw.fact_forecast
    GROUP BY metric, department_id
    ORDER BY metric, department_id
""").fetchdf().to_string(index=False))

In [ ]:
display(con.execute("""
    SELECT
        COUNT(*) AS total_rows,

        COUNT(DISTINCT (
            metric,
            department_id,
            forecast_month
        )) AS unique_forecast_combinations,

        COUNT(*) - COUNT(DISTINCT (
            metric,
            department_id,
            forecast_month
        )) AS duplicate_forecast_combinations,

        COUNT(*) FILTER (
            WHERE NOT regexp_matches(
                trained_through,
                '^[0-9]{4}-(0[1-9]|1[0-2])$'
            )
        ) AS invalid_training_cutoff_formats

    FROM raw.fact_forecast
""").fetchdf().to_string(index=False))

display(con.execute("""
    SELECT
        trained_through,
        COUNT(*) AS record_count
    FROM raw.fact_forecast
    GROUP BY trained_through
    ORDER BY trained_through
""").fetchdf().to_string(index=False))

In [ ]:
from pathlib import Path

notes_path = Path("docs/PQ-01_contract_notes.md")

forecast_notes = """

## fact_forecast — Validation evidence

- Total input records: 72
- Contract fields checked: 9
- Missing required fields: 0
- Duplicate forecast IDs: 0
- Invalid FC-nnnn ID formats: 0
- Invalid metric values: 0
- Invalid department references: 0
- Forecast months not on the first day of the month: 0
- Negative forecast values: 0
- Invalid lower confidence bounds: 0
- Invalid upper confidence bounds: 0
- Invalid trained_through formats: 0
- Duplicate metric/department/month combinations: 0
- Training cut-off: 2026-09 for all 72 records
- Forecast period: October 2026 through March 2027
- Additional physical columns: model_version, forecast_month_key
- Raw source data modified: No

Status: All checked contract rules passed. Additional physical
columns were observed and have not been treated as violations.
"""

if not notes_path.exists():
    raise FileNotFoundError(
        f"Notes file not found: {notes_path.resolve()}"
    )

existing_notes = notes_path.read_text(encoding="utf-8")

if "## fact_forecast — Validation evidence" in existing_notes:
    print("Forecast evidence section already exists; no duplicate added.")
else:
    with notes_path.open("a", encoding="utf-8") as f:
        f.write(forecast_notes)

    print("Forecast evidence saved:", notes_path.resolve())

print("File exists:", notes_path.exists())

In [ ]:
display(con.execute("""
    SELECT
        feed,
        column_name,
        data_type,
        nullable,
        rule_or_allowed_values,
        description
    FROM raw.data_contracts
    WHERE LOWER(feed) LIKE '%task%'
    ORDER BY column_name
""").fetchdf().to_string(index=False))

In [ ]:
print("TASK TABLE STRUCTURE")

print(con.execute("""
    DESCRIBE raw.fact_task
""").fetchdf().to_string(index=False))

print("\nTOTAL TASK RECORDS")

print(con.execute("""
    SELECT COUNT(*) AS total_rows
    FROM raw.fact_task
""").fetchdf().to_string(index=False))

In [ ]:
display(con.execute("""
    SELECT
        COUNT(*) AS total_rows,

        COUNT(*) FILTER (
            WHERE task_id IS NULL OR TRIM(task_id) = ''
        ) AS missing_task_id,

        COUNT(*) FILTER (
            WHERE team_id IS NULL OR TRIM(team_id) = ''
        ) AS missing_team_id,

        COUNT(*) FILTER (
            WHERE sprint_id IS NULL OR TRIM(sprint_id) = ''
        ) AS missing_sprint_id,

        COUNT(*) FILTER (
            WHERE assignee_employee_id IS NULL
               OR TRIM(assignee_employee_id) = ''
        ) AS missing_assignee,

        COUNT(*) FILTER (
            WHERE task_type IS NULL OR TRIM(task_type) = ''
        ) AS missing_task_type,

        COUNT(*) FILTER (
            WHERE priority IS NULL OR TRIM(priority) = ''
        ) AS missing_priority,

        COUNT(*) FILTER (
            WHERE story_points IS NULL
        ) AS missing_story_points,

        COUNT(*) FILTER (
            WHERE created_date IS NULL
        ) AS missing_created_date,

        COUNT(*) FILTER (
            WHERE due_date IS NULL
        ) AS missing_due_date,

        COUNT(*) FILTER (
            WHERE status IS NULL OR TRIM(status) = ''
        ) AS missing_status,

        COUNT(*) FILTER (
            WHERE reopened_count IS NULL
        ) AS missing_reopened_count,

        COUNT(*) FILTER (
            WHERE started_date IS NULL
              AND status <> 'To Do'
        ) AS missing_started_date_outside_todo,

        COUNT(*) FILTER (
            WHERE status = 'Done'
              AND completed_date IS NULL
        ) AS done_tasks_missing_completed_date

    FROM raw.fact_task
""").fetchdf().to_string(index=False))

In [ ]:
display(con.execute("""
    SELECT
        task_id,
        status,
        created_date,
        started_date,
        due_date,
        completed_date,
        sprint_id,
        assignee_employee_id,
        story_points
    FROM raw.fact_task
    WHERE status = 'Done'
      AND completed_date IS NULL
    ORDER BY task_id
""").fetchdf().to_string(index=False))

In [ ]:
display(con.execute("""
    SELECT
        COUNT(*) AS total_rows,

        COUNT(*) FILTER (
            WHERE started_date IS NOT NULL
              AND created_date > started_date
        ) AS created_after_started,

        COUNT(*) FILTER (
            WHERE completed_date IS NOT NULL
              AND started_date IS NOT NULL
              AND completed_date < started_date
        ) AS completed_before_started,

        COUNT(*) FILTER (
            WHERE started_date IS NOT NULL
              AND due_date < started_date
        ) AS due_before_started

    FROM raw.fact_task
""").fetchdf().to_string(index=False))

In [ ]:
display(con.execute("""
    SELECT
        task_id,
        status,
        created_date,
        started_date,
        completed_date,
        due_date,
        sprint_id,
        assignee_employee_id
    FROM raw.fact_task
    WHERE completed_date IS NOT NULL
      AND started_date IS NOT NULL
      AND completed_date < started_date
    ORDER BY task_id
""").fetchdf().to_string(index=False))

In [ ]:
display(con.execute("""
    SELECT
        COUNT(*) AS total_rows,

        COUNT(*) FILTER (
            WHERE task_id IS NULL
               OR NOT regexp_matches(task_id, '^TASK-[0-9]{5}$')
        ) AS invalid_task_id_format,

        COUNT(DISTINCT task_id) AS distinct_task_ids,

        COUNT(*) FILTER (
            WHERE priority NOT IN ('P1', 'P2', 'P3', 'P4')
               OR priority IS NULL
        ) AS invalid_priority,

        COUNT(*) FILTER (
            WHERE story_points NOT IN (1, 2, 3, 5, 8, 13)
               OR story_points IS NULL
        ) AS invalid_story_points,

        COUNT(*) FILTER (
            WHERE status NOT IN (
                'Done', 'In Progress', 'Blocked', 'To Do', 'Cancelled'
            ) OR status IS NULL
        ) AS invalid_status,

        COUNT(*) FILTER (
            WHERE task_type NOT IN ('Feature', 'Bug', 'Chore', 'Spike')
               OR task_type IS NULL
        ) AS invalid_task_type,

        COUNT(*) FILTER (
            WHERE reopened_count < 0 OR reopened_count IS NULL
        ) AS invalid_reopened_count

    FROM raw.fact_task
""").fetchdf().to_string(index=False))

In [ ]:
display(con.execute("""
    SELECT
        COUNT(*) AS total_rows,

        COUNT(*) FILTER (
            WHERE e.employee_id IS NULL
        ) AS unmatched_employee_ids,

        COUNT(*) FILTER (
            WHERE tm.team_id IS NULL
        ) AS unmatched_team_ids,

        COUNT(*) FILTER (
            WHERE s.sprint_id IS NULL
        ) AS unmatched_sprint_ids,

        COUNT(*) FILTER (
            WHERE s.sprint_id IS NOT NULL
              AND CAST(t.due_date AS DATE)
                  <> CAST(s.end_date AS DATE)
        ) AS due_date_sprint_end_mismatches

    FROM raw.fact_task t

    LEFT JOIN (
        SELECT DISTINCT employee_id
        FROM raw.dim_employee
    ) e
        ON t.assignee_employee_id = e.employee_id

    LEFT JOIN (
        SELECT DISTINCT team_id
        FROM raw.dim_team
    ) tm
        ON t.team_id = tm.team_id

    LEFT JOIN raw.dim_sprint s
        ON t.sprint_id = s.sprint_id
""").fetchdf().to_string(index=False))

In [ ]:
display(con.execute("""
    SELECT
        status,
        COUNT(*) AS total_tasks,
        COUNT(*) FILTER (
            WHERE started_date IS NULL
        ) AS missing_started_date,
        COUNT(*) FILTER (
            WHERE completed_date IS NULL
        ) AS missing_completed_date
    FROM raw.fact_task
    GROUP BY status
    ORDER BY status
""").fetchdf().to_string(index=False))

In [ ]:
display(con.execute("""
    SELECT
        COUNT(*) AS total_tasks,

        COUNT(*) FILTER (
            WHERE status = 'Done'
              AND completed_date IS NULL
        ) AS done_missing_completion,

        COUNT(*) FILTER (
            WHERE completed_date < started_date
              AND completed_date IS NOT NULL
              AND started_date IS NOT NULL
        ) AS completion_before_start,

        COUNT(*) FILTER (
            WHERE status = 'To Do'
              AND started_date IS NOT NULL
        ) AS todo_with_started_date,

        COUNT(*) FILTER (
            WHERE due_date <> s.end_date
        ) AS sprint_due_date_mismatches

    FROM raw.fact_task t
    LEFT JOIN raw.dim_sprint s
        ON t.sprint_id = s.sprint_id
""").fetchdf().to_string(index=False))

In [ ]:
from pathlib import Path
from datetime import datetime

notes_path = Path.cwd() / "docs" / "PQ-01_contract_notes.md"

task_evidence = """

## fact_task — Validation Evidence

Validation performed against `raw.fact_task` and its reference tables.
Raw source data was not modified.

- Total records checked: 28,868.
- Task IDs: 28,868 distinct IDs; all match `TASK-` followed by five digits.
- Required fields: no missing mandatory values in the checked fields.
- Allowed values: priority, story points, status, and task type passed.
- Reopened count: no invalid negative or missing values.
- References: no unmatched employee, team, or sprint IDs.
- Sprint due-date rule: zero mismatches.
- Created date after started date: zero violations.
- Started date missing outside `To Do`: zero violations.
- Due date before started date: zero cases in the plausibility check.
- `Done` tasks missing completed date: 8 records require investigation.
- Completed date earlier than started date: 12 records require investigation.
- Additional physical columns: `cycle_time_days`, `is_overdue_at_cutoff`,
  and `completed_on_time`. These are not listed as contract fields.

The 8 missing completion dates and 12 date-order violations must be handled
by the validation/quarantine process. No source records were changed.
"""

existing = notes_path.read_text(encoding="utf-8")

if "## fact_task — Validation Evidence" in existing:
    print("Task evidence section already exists. No duplicate section added.")
else:
    with notes_path.open("a", encoding="utf-8") as f:
        f.write(task_evidence)
    print("Task validation evidence appended.")

print("Documentation path:", notes_path.resolve())
print("File exists:", notes_path.exists())

In [ ]:
from pathlib import Path

etl_path = Path.cwd() / "src" / "etl.py"

print("ETL file exists:", etl_path.exists())
print("ETL file path:", etl_path.resolve())

if etl_path.exists():
    etl_code = etl_path.read_text(encoding="utf-8")
    print("\n--- EXISTING ETL CODE ---\n")
    print(etl_code)
else:
    print("Check that src/etl.py exists in your project folder.")

In [ ]:
display(con.execute("""
    DESCRIBE raw.fact_payroll
""").fetchdf().to_string(index=False))

display(con.execute("""
    DESCRIBE raw.dim_employee
""").fetchdf().to_string(index=False))

In [ ]:
display(con.execute("""
    WITH duplicate_runs AS (
        SELECT
            employee_key,
            run_id,
            COUNT(*) AS record_count
        FROM raw.fact_payroll
        GROUP BY employee_key, run_id
        HAVING COUNT(*) > 1
    ),
    payroll_checks AS (
        SELECT
            p.*,
            CASE
                WHEN p.employee_key IS NULL
                  OR e.employee_key IS NULL
                THEN 1 ELSE 0
            END AS unmatched_employee_key
        FROM raw.fact_payroll p
        LEFT JOIN raw.dim_employee e
            ON p.employee_key = e.employee_key
    )
    SELECT
        (SELECT COUNT(*) FROM raw.fact_payroll) AS input_rows,

        (SELECT COUNT(*)
         FROM duplicate_runs) AS duplicate_employee_run_groups,

        (SELECT COALESCE(SUM(record_count - 1), 0)
         FROM duplicate_runs) AS excess_duplicate_rows,

        (SELECT COUNT(*)
         FROM payroll_checks
         WHERE unmatched_employee_key = 1)
            AS unmatched_employee_key_rows,

        (SELECT COUNT(*)
         FROM raw.fact_payroll
         WHERE payroll_item_id IS NULL)
            AS missing_payroll_item_ids

""").fetchdf().to_string(index=False))

In [ ]:
display(con.execute("""
    SELECT
        payroll_item_id,
        run_id,
        employee_id,
        employee_key,
        period_end_date,
        gross_pay,
        net_pay,
        duplicate_count
    FROM (
        SELECT
            p.*,
            COUNT(*) OVER (
                PARTITION BY employee_key, run_id
            ) AS duplicate_count
        FROM raw.fact_payroll p
    ) d
    WHERE duplicate_count > 1
    ORDER BY employee_key, run_id, payroll_item_id
""").fetchdf().to_string(index=False))

In [ ]:
display(con.execute("""
    SELECT
        p.payroll_item_id,
        p.employee_id,
        p.employee_key,
        p.run_id,
        p.period_end_date,
        p.gross_pay,
        p.net_pay,
        e.employee_id AS matched_employee_id,
        e.employee_key AS matched_employee_key
    FROM raw.fact_payroll p
    LEFT JOIN raw.dim_employee e
        ON p.employee_key = e.employee_key
    WHERE e.employee_key IS NULL
    ORDER BY p.payroll_item_id
""").fetchdf().to_string(index=False))

In [ ]:
display(con.execute("""
    WITH duplicate_groups AS (
        SELECT
            employee_key,
            run_id,
            COUNT(*) AS group_size,
            COUNT(*) FILTER (
                WHERE payroll_item_id LIKE 'PI-X%'
            ) AS pi_x_rows,
            COUNT(*) FILTER (
                WHERE payroll_item_id NOT LIKE 'PI-X%'
            ) AS normal_id_rows
        FROM raw.fact_payroll
        GROUP BY employee_key, run_id
        HAVING COUNT(*) > 1
    )
    SELECT
        COUNT(*) AS duplicate_groups,
        COUNT(*) FILTER (
            WHERE pi_x_rows = 1
              AND normal_id_rows = 1
              AND group_size = 2
        ) AS groups_with_one_normal_and_one_pi_x,
        COUNT(*) FILTER (
            WHERE NOT (
                pi_x_rows = 1
                AND normal_id_rows = 1
                AND group_size = 2
            )
        ) AS other_duplicate_patterns
    FROM duplicate_groups
""").fetchdf().to_string(index=False))

In [ ]:
from pathlib import Path

etl_v2_path = Path.cwd() / "src" / "etl_v2.py"

etl_v2_code = r'''import json
from pathlib import Path

import duckdb
import pandas as pd

ROOT = Path(__file__).resolve().parent.parent
DB_PATH = ROOT / "05_Starter" / "pulseiq.duckdb"
OUTPUT_DIR = ROOT / "outputs" / "pq01"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def run_payroll_etl():
    con = duckdb.connect(str(DB_PATH), read_only=True)
    try:
        payroll = con.execute(
            "SELECT * FROM raw.fact_payroll"
        ).fetchdf()

        employees = con.execute(
            "SELECT DISTINCT employee_key FROM raw.dim_employee"
        ).fetchdf()
    finally:
        con.close()

    # Validate employee references using the historical employee key.
    valid_employee_keys = set(
        employees["employee_key"].dropna().tolist()
    )

    payroll["quarantine_reasons"] = [[] for _ in range(len(payroll))]

    unmatched_mask = (
        payroll["employee_key"].isna()
        | ~payroll["employee_key"].isin(valid_employee_keys)
    )

    for idx in payroll.index[unmatched_mask]:
        payroll.at[idx, "quarantine_reasons"].append(
            "UNMATCHED_EMPLOYEE_KEY"
        )

    # Detect duplicate employee-run groups dynamically.
    duplicate_mask = payroll.duplicated(
        subset=["employee_key", "run_id"],
        keep=False
    )

    duplicate_rows = payroll.loc[duplicate_mask].copy()

    if not duplicate_rows.empty:
        group_sizes = duplicate_rows.groupby(
            ["employee_key", "run_id"],
            dropna=False
        ).size()

        pi_x_counts = duplicate_rows.assign(
            is_pi_x=duplicate_rows["payroll_item_id"]
            .astype("string")
            .str.startswith("PI-X", na=False)
        ).groupby(
            ["employee_key", "run_id"],
            dropna=False
        )["is_pi_x"].sum()

        normal_counts = group_sizes - pi_x_counts

        unexpected = (
            (group_sizes != 2)
            | (pi_x_counts != 1)
            | (normal_counts != 1)
        )

        if unexpected.any():
            bad_groups = unexpected[unexpected].index.tolist()
            raise ValueError(
                "Unexpected duplicate patterns found; "
                f"pipeline stopped safely. Groups: {bad_groups}"
            )

        # Keep the non-PI-X record in each validated duplicate pair.
        pi_x_mask = (
            payroll["payroll_item_id"]
            .astype("string")
            .str.startswith("PI-X", na=False)
        )

        duplicate_quarantine_mask = duplicate_mask & pi_x_mask

        for idx in payroll.index[duplicate_quarantine_mask]:
            payroll.at[idx, "quarantine_reasons"].append(
                "DUPLICATE_EMPLOYEE_RUN"
            )

    # Serialize reasons so every failure remains visible in output.
    payroll["quarantine_reason"] = payroll[
        "quarantine_reasons"
    ].apply(lambda reasons: "|".join(reasons) if reasons else None)

    quarantine = payroll[
        payroll["quarantine_reason"].notna()
    ].copy()

    passed = payroll[
        payroll["quarantine_reason"].isna()
    ].copy()

    input_count = len(payroll)
    passed_count = len(passed)
    quarantine_count = len(quarantine)

    if input_count != passed_count + quarantine_count:
        raise AssertionError("Input/pass/quarantine counts do not reconcile.")

    # Remove internal helper column from the business-data outputs.
    helper_columns = ["quarantine_reasons"]

    quarantine.drop(columns=helper_columns).to_csv(
        OUTPUT_DIR / "fact_payroll_quarantine.csv",
        index=False
    )
    passed.drop(
        columns=helper_columns + ["quarantine_reason"]
    ).to_csv(
        OUTPUT_DIR / "fact_payroll_passed.csv",
        index=False
    )

    summary = {
        "table": "fact_payroll",
        "rows_in": input_count,
        "rows_passed": passed_count,
        "rows_quarantined": quarantine_count,
        "reconciliation_ok": (
            input_count == passed_count + quarantine_count
        ),
        "quarantine_reasons": {
            str(reason): int(count)
            for reason, count in (
                quarantine["quarantine_reason"]
                .str.split("|")
                .explode()
                .value_counts()
                .items()
            )
        },
    }

    with open(
        OUTPUT_DIR / "fact_payroll_summary.json",
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(summary, f, indent=2)

    print(json.dumps(summary, indent=2))
    print(f"\nOutput folder: {OUTPUT_DIR}")


if __name__ == "__main__":
    run_payroll_etl()
'''

etl_v2_path.write_text(etl_v2_code, encoding="utf-8")

print("Created:", etl_v2_path.resolve())
print("File exists:", etl_v2_path.exists())
print("Existing etl.py was not changed.")

In [ ]:
import ast
from pathlib import Path

etl_v2_path = Path.cwd() / "src" / "etl_v2.py"

code = etl_v2_path.read_text(encoding="utf-8-sig")

try:
    ast.parse(code)
    print("Syntax check: PASSED")
except SyntaxError as e:
    print("Syntax check: FAILED")
    print("Line:", e.lineno)
    print("Error:", e.msg)

print("Script exists:", etl_v2_path.exists())
print("Script size:", etl_v2_path.stat().st_size, "bytes")

In [ ]:
from pathlib import Path

etl_v2_path = Path.cwd() / "src" / "etl_v2.py"
code = etl_v2_path.read_text(encoding="utf-8-sig")

checks = {
    "Read-only database connection": "read_only=True" in code,
    "Dynamic duplicate detection": "duplicated(" in code,
    "Employee-key validation": "UNMATCHED_EMPLOYEE_KEY" in code,
    "Duplicate quarantine reason": "DUPLICATE_EMPLOYEE_RUN" in code,
    "Reconciliation check": "reconciliation_ok" in code,
    "Unexpected-pattern safety check": "Unexpected duplicate patterns found" in code,
    "No hardcoded PI-X record IDs": (
        '"PI-X0001"' not in code
        and '"PI-X0002"' not in code
        and '"PI-X0003"' not in code
        and '"PI-X0004"' not in code
    ),
}

for name, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'} - {name}")

print("\nPreflight passed:", all(checks.values()))

In [ ]:
import subprocess
import sys
from pathlib import Path

etl_v2_path = Path.cwd() / "src" / "etl_v2.py"

result = subprocess.run(
    [sys.executable, str(etl_v2_path)],
    capture_output=True,
    text=True,
    timeout=120
)

print("Return code:", result.returncode)

if result.stdout:
    print("\n--- ETL OUTPUT ---")
    print(result.stdout)

if result.stderr:
    print("\n--- ERRORS / WARNINGS ---")
    print(result.stderr)

if result.returncode != 0:
    print("\nETL FAILED. Do not treat the task as completed.")
else:
    print("\nETL execution completed. Output validation is still required.")

In [ ]:
import json
import pandas as pd
from pathlib import Path

output_dir = Path.cwd() / "outputs" / "pq01"

passed_path = output_dir / "fact_payroll_passed.csv"
quarantine_path = output_dir / "fact_payroll_quarantine.csv"
summary_path = output_dir / "fact_payroll_summary.json"

print("Output file checks:")
for path in [passed_path, quarantine_path, summary_path]:
    print(f"{path.name}: exists={path.exists()}")

passed_df = pd.read_csv(passed_path)
quarantine_df = pd.read_csv(quarantine_path)

with open(summary_path, encoding="utf-8") as f:
    summary = json.load(f)

print("\nPassed CSV rows:", len(passed_df))
print("Quarantine CSV rows:", len(quarantine_df))
print("Summary input rows:", summary["rows_in"])
print("Reconciliation:", summary["reconciliation_ok"])

print("\nQuarantined records and reasons:")
print(
    quarantine_df[
        ["payroll_item_id", "employee_id", "employee_key",
         "run_id", "quarantine_reason"]
    ].to_string(index=False)
)

In [ ]:
import hashlib
import subprocess
import sys
from pathlib import Path

output_dir = Path.cwd() / "outputs" / "pq01"
etl_path = Path.cwd() / "src" / "etl_v2.py"

files_to_check = [
    output_dir / "fact_payroll_passed.csv",
    output_dir / "fact_payroll_quarantine.csv",
    output_dir / "fact_payroll_summary.json",
]

def file_hash(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

# Capture the first-run output fingerprints.
before = {
    path.name: file_hash(path)
    for path in files_to_check
}

# Run the ETL a second time.
result = subprocess.run(
    [sys.executable, str(etl_path)],
    capture_output=True,
    text=True,
    timeout=120
)

print("Second-run return code:", result.returncode)

if result.stdout:
    print("\nSecond-run summary:")
    print(result.stdout)

# Compare output fingerprints.
after = {
    path.name: file_hash(path)
    for path in files_to_check
}

print("\nRepeatability results:")
for filename in before:
    print(
        f"{filename}: "
        f"{'IDENTICAL' if before[filename] == after[filename] else 'CHANGED'}"
    )

print(
    "\nRepeatability passed:",
    result.returncode == 0 and before == after
)

In [ ]:
import duckdb
from pathlib import Path

db_path = Path.cwd() / "05_Starter" / "pulseiq.duckdb"

check_con = duckdb.connect(str(db_path), read_only=True)

try:
    raw_counts = check_con.execute("""
        SELECT 'fact_payroll' AS table_name, COUNT(*) AS row_count
        FROM raw.fact_payroll

        UNION ALL

        SELECT 'dim_employee' AS table_name, COUNT(*) AS row_count
        FROM raw.dim_employee

        ORDER BY table_name
    """).fetchdf()

    print("Raw source table counts:")
    print(raw_counts.to_string(index=False))

finally:
    check_con.close()

print("\nExpected baseline:")
print("fact_payroll: 7344")
print("dim_employee: 1606")

In [ ]:
from pathlib import Path
import json

root = Path.cwd()
notes_path = root / "docs" / "PQ-01_contract_notes.md"
summary_path = root / "outputs" / "pq01" / "fact_payroll_summary.json"

with open(summary_path, encoding="utf-8") as f:
    summary = json.load(f)

etl_evidence = f"""

## Payroll ETL — Execution and Test Evidence

- Script: `src/etl_v2.py`
- Source: `raw.fact_payroll` and `raw.dim_employee`
- Source connection: DuckDB read-only mode.
- Input rows: {summary["rows_in"]}.
- Passed rows: {summary["rows_passed"]}.
- Quarantined rows: {summary["rows_quarantined"]}.
- Reconciliation passed: {summary["reconciliation_ok"]}.
- Quarantine reasons: {json.dumps(summary["quarantine_reasons"])}.
- Output files: `fact_payroll_passed.csv`,
  `fact_payroll_quarantine.csv`, and `fact_payroll_summary.json`.
- Repeatability: two consecutive executions produced identical
  output-file SHA-256 fingerprints.
- Source row-count checks after execution:
  `raw.fact_payroll` = 7,344 and `raw.dim_employee` = 1,606.
- Limitation: matching row counts do not establish full row-level
  source immutability. No intentional writes to raw tables were performed.

The original `src/etl.py` prototype remains unchanged. The replacement
prototype is `src/etl_v2.py`.
"""

existing = notes_path.read_text(encoding="utf-8")

if "## Payroll ETL — Execution and Test Evidence" in existing:
    print("Payroll ETL evidence section already exists; no duplicate added.")
else:
    with notes_path.open("a", encoding="utf-8") as f:
        f.write(etl_evidence)
    print("Payroll ETL evidence appended successfully.")

print("Notes file exists:", notes_path.exists())
print("Notes path:", notes_path.resolve())

In [ ]:
display(con.execute("""
    SELECT *
    FROM raw.data_contracts
""").fetchdf().to_string(index=False))

In [ ]:
contracts_df = con.execute("""
    SELECT
        feed,
        column_name,
        data_type,
        nullable,
        rule_or_allowed_values
    FROM raw.data_contracts
    ORDER BY feed, column_name
""").fetchdf()

print("Contract rows:", len(contracts_df))
print("\nContracts by feed:")
print(
    contracts_df.groupby("feed")
    .size()
    .rename("contract_rules")
    .to_string()
)

print("\nRequired fields by feed:")
print(
    contracts_df[contracts_df["nullable"] == False]
    .groupby("feed")
    .size()
    .rename("non_nullable_contract_fields")
    .to_string()
)

In [ ]:
feed_tables = [
    "fact_anomaly",
    "fact_attrition_risk",
    "fact_forecast",
    "fact_task",
]

feed_counts = []

for table in feed_tables:
    count = con.execute(
        f"SELECT COUNT(*) FROM raw.{table}"
    ).fetchone()[0]

    feed_counts.append({
        "feed": table,
        "input_rows": count,
    })

feed_counts_df = pd.DataFrame(feed_counts)

print("Baseline row counts:")
print(feed_counts_df.to_string(index=False))

print("\nTotal rows across contracted feeds:",
      feed_counts_df["input_rows"].sum())

In [ ]:
from pathlib import Path

output_dir = Path.cwd() / "outputs" / "pq01"

print("Output directory:", output_dir.resolve())
print("\nCurrent files:")

for path in sorted(output_dir.iterdir()):
    if path.is_file():
        print(f"{path.name} — {path.stat().st_size:,} bytes")

In [ ]:
tables_to_inspect = [
    "fact_anomaly",
    "fact_attrition_risk",
    "fact_forecast",
    "fact_task",
]

for table in tables_to_inspect:
    print(f"\n--- raw.{table} ---")
    schema = con.execute(
        f"DESCRIBE raw.{table}"
    ).fetchdf()

    print(schema[["column_name", "column_type"]].to_string(index=False))

In [ ]:
display(con.execute("""
    SELECT
        'fact_anomaly' AS feed,
        COUNT(*) AS input_rows,
        COUNT(*) FILTER (
            WHERE result_id IS NULL
               OR NOT regexp_matches(result_id, '^AN-[0-9]{4}$')
        ) AS invalid_id_rows,
        COUNT(*) FILTER (
            WHERE anomaly_score IS NULL
               OR anomaly_score < 0
               OR anomaly_score > 1
        ) AS invalid_score_rows
    FROM raw.fact_anomaly

    UNION ALL

    SELECT
        'fact_attrition_risk',
        COUNT(*),
        COUNT(*) FILTER (
            WHERE prediction_id IS NULL
               OR NOT regexp_matches(prediction_id, '^AR-[0-9]{4}$')
        ),
        COUNT(*) FILTER (
            WHERE risk_score_12m IS NULL
               OR risk_score_12m < 0
               OR risk_score_12m > 1
        )
    FROM raw.fact_attrition_risk

    UNION ALL

    SELECT
        'fact_forecast',
        COUNT(*),
        COUNT(*) FILTER (
            WHERE forecast_id IS NULL
               OR NOT regexp_matches(forecast_id, '^FC-[0-9]{4}$')
        ),
        COUNT(*) FILTER (
            WHERE forecast_value IS NULL
               OR forecast_value < 0
        )
    FROM raw.fact_forecast
""").fetchdf().to_string(index=False))

In [ ]:
display(con.execute("""
    SELECT
        COUNT(*) AS total_rows,

        COUNT(DISTINCT result_id) AS distinct_result_ids,

        COUNT(*) FILTER (
            WHERE result_id IS NULL
               OR source_dataset IS NULL
               OR record_id IS NULL
               OR employee_id IS NULL
               OR period IS NULL
               OR anomaly_score IS NULL
               OR flagged IS NULL
               OR reason_code IS NULL
               OR reason_text IS NULL
               OR model_version IS NULL
               OR scored_at IS NULL
        ) AS rows_with_missing_required_fields,

        COUNT(*) FILTER (
            WHERE flagged NOT IN ('Y', 'N')
        ) AS invalid_flag_values,

        COUNT(*) FILTER (
            WHERE source_dataset NOT IN (
                'attendance_monthly_features',
                'payroll_monthly_features'
            )
        ) AS invalid_source_dataset,

        COUNT(*) FILTER (
            WHERE e.employee_id IS NULL
        ) AS unmatched_employee_ids,

        COUNT(*) FILTER (
            WHERE source_dataset = 'attendance_monthly_features'
              AND money_at_risk IS NOT NULL
        ) AS attendance_money_at_risk_populated,

        COUNT(*) FILTER (
            WHERE source_dataset = 'payroll_monthly_features'
              AND money_at_risk IS NULL
        ) AS payroll_money_at_risk_missing

    FROM raw.fact_anomaly a
    LEFT JOIN (
        SELECT DISTINCT employee_id
        FROM raw.dim_employee
    ) e
        ON a.employee_id = e.employee_id
""").fetchdf().to_string(index=False))

In [ ]:
display(con.execute("""
    SELECT
        COUNT(*) AS total_rows,

        COUNT(DISTINCT a.result_id) AS distinct_result_ids,

        COUNT(*) FILTER (
            WHERE a.result_id IS NULL
               OR a.source_dataset IS NULL
               OR a.record_id IS NULL
               OR a.employee_id IS NULL
               OR a.period IS NULL
               OR a.anomaly_score IS NULL
               OR a.flagged IS NULL
               OR a.reason_code IS NULL
               OR a.reason_text IS NULL
               OR a.model_version IS NULL
               OR a.scored_at IS NULL
        ) AS rows_with_missing_required_fields,

        COUNT(*) FILTER (
            WHERE a.flagged NOT IN ('Y', 'N')
               OR a.flagged IS NULL
        ) AS invalid_flag_values,

        COUNT(*) FILTER (
            WHERE a.source_dataset NOT IN (
                'attendance_monthly_features',
                'payroll_monthly_features'
            )
               OR a.source_dataset IS NULL
        ) AS invalid_source_dataset,

        COUNT(*) FILTER (
            WHERE e.employee_id IS NULL
        ) AS unmatched_employee_ids,

        COUNT(*) FILTER (
            WHERE a.source_dataset = 'attendance_monthly_features'
              AND a.money_at_risk IS NOT NULL
        ) AS attendance_money_at_risk_populated,

        COUNT(*) FILTER (
            WHERE a.source_dataset = 'payroll_monthly_features'
              AND a.money_at_risk IS NULL
        ) AS payroll_money_at_risk_missing

    FROM raw.fact_anomaly a
    LEFT JOIN (
        SELECT DISTINCT employee_id
        FROM raw.dim_employee
    ) e
        ON a.employee_id = e.employee_id
""").fetchdf().to_string(index=False))

In [ ]:
display(con.execute("""
    SELECT
        MIN(anomaly_score) AS min_anomaly_score,
        MAX(anomaly_score) AS max_anomaly_score,
        COUNT(*) FILTER (
            WHERE anomaly_score < 0 OR anomaly_score > 1
        ) AS scores_outside_0_to_1,
        COUNT(DISTINCT reason_code) AS distinct_reason_codes,
        COUNT(DISTINCT model_version) AS distinct_model_versions
    FROM raw.fact_anomaly
""").fetchdf().to_string(index=False))

display(con.execute("""
    SELECT
        source_dataset,
        model_version,
        COUNT(*) AS record_count,
        COUNT(*) FILTER (
            WHERE money_at_risk < 0
        ) AS negative_money_at_risk
    FROM raw.fact_anomaly
    GROUP BY source_dataset, model_version
    ORDER BY source_dataset, model_version
""").fetchdf().to_string(index=False))

In [ ]:
display(con.execute("""
    SELECT
        a.source_dataset,
        COUNT(*) AS total_anomalies,

        COUNT(*) FILTER (
            WHERE p.payroll_item_id IS NOT NULL
        ) AS matched_payroll_records,

        COUNT(*) FILTER (
            WHERE att.attendance_record_id IS NOT NULL
        ) AS matched_attendance_records

    FROM raw.fact_anomaly AS a

    LEFT JOIN raw.fact_payroll AS p
        ON a.record_id = p.payroll_item_id

    LEFT JOIN raw.fact_attendance_monthly AS att
        ON a.record_id = att.attendance_record_id

    GROUP BY a.source_dataset
    ORDER BY a.source_dataset
""").fetchdf().to_string(index=False))

In [ ]:
display(con.execute("""
    SELECT
        a.source_dataset,
        COUNT(*) AS total_anomalies,

        COUNT(*) FILTER (
            WHERE p.payroll_item_id IS NOT NULL
        ) AS matched_payroll_records,

        COUNT(*) FILTER (
            WHERE att.attendance_fact_id IS NOT NULL
        ) AS matched_attendance_records

    FROM raw.fact_anomaly AS a

    LEFT JOIN raw.fact_payroll AS p
        ON a.record_id = p.payroll_item_id

    LEFT JOIN raw.fact_attendance_monthly AS att
        ON a.record_id = att.attendance_fact_id

    GROUP BY a.source_dataset
    ORDER BY a.source_dataset
""").fetchdf().to_string(index=False))

In [ ]:
print("ANOMALY ATTENDANCE RECORD IDs")
display(con.execute("""
    SELECT record_id, employee_id, period
    FROM raw.fact_anomaly
    WHERE source_dataset = 'attendance_monthly_features'
    LIMIT 10
""").fetchdf())

print("ATTENDANCE TABLE ID COLUMNS")
display(con.execute("""
    SELECT
        attendance_fact_id,
        employee_id,
        month
    FROM raw.fact_attendance_monthly
    LIMIT 10
""").fetchdf())

In [ ]:
display(con.execute("""
    DESCRIBE raw.fact_attendance_monthly
""").fetchdf().to_string(index=False))

In [ ]:
display(con.execute("""
    SELECT
        COUNT(*) AS total_attendance_anomalies,

        COUNT(*) FILTER (
            WHERE m.match_count = 1
        ) AS uniquely_matched,

        COUNT(*) FILTER (
            WHERE m.match_count = 0
        ) AS unmatched,

        COUNT(*) FILTER (
            WHERE m.match_count > 1
        ) AS multiple_matches

    FROM raw.fact_anomaly AS a

    LEFT JOIN (
        SELECT
            employee_id,
            DATE_TRUNC('month', period_end_date)::DATE AS attendance_month,
            COUNT(*) AS match_count
        FROM raw.fact_attendance_monthly
        GROUP BY
            employee_id,
            DATE_TRUNC('month', period_end_date)::DATE
    ) AS m
        ON a.employee_id = m.employee_id
       AND DATE_TRUNC('month', a.period)::DATE = m.attendance_month

    WHERE a.source_dataset = 'attendance_monthly_features'
""").fetchdf().to_string(index=False))

In [ ]:
display(con.execute("""
    SELECT
        COUNT(*) AS total_attendance_anomalies,

        COUNT(*) FILTER (
            WHERE EXTRACT(DAY FROM a.period) <> 1
        ) AS periods_not_first_of_month,

        COUNT(*) FILTER (
            WHERE DATE_TRUNC('month', a.period)::DATE
                  <> DATE_TRUNC('month', att.period_end_date)::DATE
        ) AS employee_month_mismatches,

        COUNT(*) FILTER (
            WHERE att.period_end_date IS NULL
        ) AS missing_attendance_period_end

    FROM raw.fact_anomaly AS a

    LEFT JOIN raw.fact_attendance_monthly AS att
        ON a.employee_id = att.employee_id
       AND DATE_TRUNC('month', a.period)::DATE
           = DATE_TRUNC('month', att.period_end_date)::DATE

    WHERE a.source_dataset = 'attendance_monthly_features'
""").fetchdf().to_string(index=False))

In [ ]:
anomaly_summary = con.execute("""
    SELECT
        a.source_dataset,
        COUNT(*) AS input_rows,
        COUNT(DISTINCT a.result_id) AS distinct_result_ids,
        COUNT(*) FILTER (
            WHERE a.anomaly_score < 0
               OR a.anomaly_score > 1
               OR a.anomaly_score IS NULL
        ) AS invalid_anomaly_scores,
        COUNT(*) FILTER (
            WHERE a.flagged IS NULL
               OR a.flagged NOT IN ('Y', 'N')
        ) AS invalid_flags,
        COUNT(*) FILTER (
            WHERE a.employee_id IS NULL
        ) AS missing_employee_ids
    FROM raw.fact_anomaly AS a
    GROUP BY a.source_dataset
    ORDER BY a.source_dataset
""").fetchdf()

display(anomaly_summary)

In [ ]:
display(con.execute("""
    SELECT
        COUNT(*) AS total_rows,

        COUNT(DISTINCT result_id) AS unique_result_ids,

        COUNT(*) - COUNT(DISTINCT result_id)
            AS duplicate_result_id_rows

    FROM raw.fact_anomaly
""").fetchdf().to_string(index=False))

display(con.execute("""
    SELECT
        source_dataset,
        record_id,
        employee_id,
        period,
        COUNT(*) AS occurrences
    FROM raw.fact_anomaly
    GROUP BY
        source_dataset,
        record_id,
        employee_id,
        period
    HAVING COUNT(*) > 1
    ORDER BY occurrences DESC
    LIMIT 20
""").fetchdf().to_string(index=False))

In [ ]:
display(con.execute("""
    SELECT
        COUNT(*) AS total_rows,

        COUNT(*) FILTER (
            WHERE scored_at IS NULL
        ) AS missing_scored_at,

        COUNT(DISTINCT scored_at) AS distinct_scored_timestamps,

        MIN(scored_at) AS earliest_scored_at,

        MAX(scored_at) AS latest_scored_at

    FROM raw.fact_anomaly
""").fetchdf().to_string(index=False))

In [ ]:
print("FLAG DISTRIBUTION")
display(con.execute("""
    SELECT
        flagged,
        COUNT(*) AS record_count
    FROM raw.fact_anomaly
    GROUP BY flagged
    ORDER BY flagged
""").fetchdf())

print("REASON CODE DISTRIBUTION")
display(con.execute("""
    SELECT
        reason_code,
        COUNT(*) AS record_count
    FROM raw.fact_anomaly
    GROUP BY reason_code
    ORDER BY record_count DESC, reason_code
""").fetchdf())

In [ ]:
display(con.execute("""
    SELECT *
    FROM raw.data_contracts
    WHERE LOWER(table_name) LIKE '%anomaly%'
       OR LOWER(field_name) IN (
            'reason_code',
            'flagged',
            'model_version'
       )
    ORDER BY table_name, field_name
""").fetchdf().to_string(index=False))

In [ ]:
display(con.execute("""
    DESCRIBE raw.data_contracts
""").fetchdf().to_string(index=False))

In [ ]:
display(con.execute("""
    SELECT
        feed,
        column_name,
        data_type,
        nullable,
        rule_or_allowed_values,
        description
    FROM raw.data_contracts
    WHERE LOWER(feed) LIKE '%anomaly%'
    ORDER BY column_name
""").fetchdf().to_string(index=False))

In [ ]:
print("REASON TEXT CHECK")
display(con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (
            WHERE reason_text IS NULL
               OR TRIM(reason_text) = ''
        ) AS missing_or_blank_reason_text
    FROM raw.fact_anomaly
""").fetchdf())

print("MODEL VERSION VALUES")
display(con.execute("""
    SELECT
        model_version,
        COUNT(*) AS record_count,
        REGEXP_MATCHES(
            model_version,
            '^[0-9]+\\.[0-9]+\\.[0-9]+$'
        ) AS strict_semver_numeric_format
    FROM raw.fact_anomaly
    GROUP BY model_version
    ORDER BY model_version
""").fetchdf())

In [ ]:
display(con.execute("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'raw'
    ORDER BY table_name
""").fetchdf().to_string(index=False))

In [ ]:
display(
    con.execute("DESCRIBE raw.data_dictionary")
    .fetchdf()
    .to_string(index=False)
)

In [ ]:
display(con.execute("""
    SELECT
        sheet,
        "column",
        data_type,
        description,
        example
    FROM raw.data_dictionary
    WHERE LOWER("column") LIKE '%reason%'
       OR LOWER(description) LIKE '%reason code%'
       OR LOWER(example) LIKE '%STATISTICAL_OUTLIER%'
    ORDER BY sheet, "column"
""").fetchdf().to_string(index=False))

In [ ]:
display(
    con.execute("DESCRIBE raw.dq_checks_catalog")
    .fetchdf()
    .to_string(index=False)
)


In [ ]:
display(con.execute("""
    SELECT
        check_id,
        check_name,
        description,
        logic_sql_or_rule,
        severity,
        tolerance,
        fix_guidance
    FROM raw.dq_checks_catalog
    WHERE LOWER(check_name) LIKE '%anomal%'
       OR LOWER(description) LIKE '%anomal%'
       OR LOWER(logic_sql_or_rule) LIKE '%reason_code%'
       OR LOWER(logic_sql_or_rule) LIKE '%model_version%'
    ORDER BY check_id
""").fetchdf().to_string(index=False))

In [ ]:
anomaly_contract_status = {
    "feed": "fact_anomaly",
    "input_rows": 405,
    "unique_result_ids": 405,
    "missing_required_fields": 0,
    "invalid_anomaly_scores": 0,
    "invalid_flags": 0,
    "invalid_source_datasets": 0,
    "unmatched_employee_ids": 0,
    "payroll_direct_id_matches": 184,
    "attendance_employee_month_matches": 221,
    "duplicate_result_ids": 0,
    "duplicate_source_employee_period_combinations": 0,
    "missing_reason_text": 0,
    "missing_scored_at": 0,
    "distinct_scored_timestamps": 1,
    "reason_code_reference": "Unresolved: complete allowed list not found",
    "model_version_semver": "Unresolved: naming convention needs clarification",
    "scored_at_timezone": "Unresolved: timezone-naive timestamp",
    "raw_source_modified": False,
}

for check, result in anomaly_contract_status.items():
    print(f"{check}: {result}")

In [ ]:
from pathlib import Path

notes_path = Path.cwd() / "docs" / "PQ-01_contract_notes.md"

anomaly_notes = """

## Fact Anomaly — Validation Evidence

- Input records: 405.
- Unique result IDs: 405; duplicate result-ID rows: 0.
- Missing required fields in the checks performed: 0.
- Invalid anomaly scores outside 0–1: 0.
- Invalid flag values: 0; all 405 records have flag Y.
- Invalid source-dataset values: 0.
- Unmatched employee IDs: 0.
- Payroll anomalies: 184/184 matched directly by payroll item ID.
- Attendance anomalies: 221/221 matched uniquely by employee and calendar month.
- Duplicate source/employee/period combinations: 0.
- Missing or blank reason text: 0.
- Missing scoring timestamps: 0; all 405 share timestamp
  2026-10-01 02:00:00.
- Negative money-at-risk values: 0 in the checks performed.
- Raw source data was not intentionally modified.

### Outstanding Contract Questions

1. The complete published reason-code allowlist was not found in the
   inspected contract, dictionary, or DQ catalog sources.
2. The model-version values are
   `iforest-attendance-0.1` and `iforest-payroll-0.1`.
   Their compliance with the intended semantic-version convention needs
   clarification; failure of a strict numeric SemVer pattern alone is
   not proof of invalidity.
3. The scoring timestamp is timezone-naive in the database, so UTC
   cannot be independently verified from the stored value.

These findings cover the checks listed above; they do not establish
that every anomaly contract requirement has passed. No records should
be quarantined solely because these reference questions remain unresolved.
"""

if not notes_path.exists():
    raise FileNotFoundError(
        f"Expected project notes file not found: {notes_path}"
    )

with notes_path.open("a", encoding="utf-8") as f:
    f.write(anomaly_notes)

print("Anomaly validation evidence appended.")
print("Documentation path:", notes_path.resolve())
print("File exists:", notes_path.exists())

In [ ]:
from pathlib import Path

project_root = Path.cwd()
etl_path = project_root / "src" / "etl_v2.py"

print("Project root:", project_root)
print("Payroll ETL exists:", etl_path.exists())

if etl_path.exists():
    print("\n--- Current etl_v2.py ---")
    print(etl_path.read_text(encoding="utf-8"))
else:
    raise FileNotFoundError(f"Expected ETL file not found: {etl_path}")

In [ ]:
feeds = [
    "fact_anomaly",
    "fact_attrition_risk",
    "fact_forecast",
    "fact_task",
]

for feed in feeds:
    print(f"\n{'=' * 20} {feed} {'=' * 20}")
    schema = con.execute(
        f"DESCRIBE raw.{feed}"
    ).fetchdf()
    display(schema[["column_name", "column_type", "null"]])

In [ ]:
reference_tables = [
    "dim_employee",
    "dim_department",
    "dim_team",
    "dim_sprint",
    "dim_date",
]

for table in reference_tables:
    print(f"\n{'=' * 15} {table} {'=' * 15}")
    display(
        con.execute(f"DESCRIBE raw.{table}")
        .fetchdf()[["column_name", "column_type"]]
    )

In [ ]:
# Reference and date preflight for the four feeds

display(con.execute("""
    SELECT
        'fact_anomaly' AS feed,
        COUNT(*) AS input_rows,
        COUNT(*) FILTER (
            WHERE a.employee_id IS NULL
               OR e.employee_id IS NULL
        ) AS employee_reference_issues
    FROM raw.fact_anomaly a
    LEFT JOIN (
        SELECT DISTINCT employee_id
        FROM raw.dim_employee
    ) e ON a.employee_id = e.employee_id

    UNION ALL

    SELECT
        'fact_attrition_risk',
        COUNT(*),
        COUNT(*) FILTER (
            WHERE a.employee_id IS NULL
               OR e.employee_id IS NULL
        )
    FROM raw.fact_attrition_risk a
    LEFT JOIN (
        SELECT DISTINCT employee_id
        FROM raw.dim_employee
    ) e ON a.employee_id = e.employee_id

    UNION ALL

    SELECT
        'fact_forecast',
        COUNT(*),
        COUNT(*) FILTER (
            WHERE a.department_id IS NOT NULL
              AND a.department_id <> 'ALL'
              AND d.department_id IS NULL
        )
    FROM raw.fact_forecast a
    LEFT JOIN raw.dim_department d
        ON a.department_id = d.department_id

    UNION ALL

    SELECT
        'fact_task',
        COUNT(*),
        COUNT(*) FILTER (
            WHERE a.assignee_employee_id IS NULL
               OR e.employee_id IS NULL
        )
    FROM raw.fact_task a
    LEFT JOIN (
        SELECT DISTINCT employee_id
        FROM raw.dim_employee
    ) e ON a.assignee_employee_id = e.employee_id
""").fetchdf().to_string(index=False))

print("\nFORECAST DATE-KEY CHECK")
display(con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (
            WHERE f.forecast_month_key IS NULL
               OR d.date_key IS NULL
        ) AS missing_or_unmatched_month_keys,
        COUNT(*) FILTER (
            WHERE f.forecast_month <> d.date
        ) AS month_key_date_mismatches
    FROM raw.fact_forecast f
    LEFT JOIN raw.dim_date d
        ON f.forecast_month_key = d.date_key
""").fetchdf().to_string(index=False))

In [ ]:
display(con.execute("""
    SELECT
        f.forecast_id,
        f.forecast_month,
        f.forecast_month_key,
        d.date AS date_from_key,
        d.month_start_date,
        d.month_end_date
    FROM raw.fact_forecast AS f
    LEFT JOIN raw.dim_date AS d
        ON f.forecast_month_key = d.date_key
    WHERE d.date_key IS NULL
       OR f.forecast_month <> d.date
    ORDER BY f.forecast_month, f.forecast_id
    LIMIT 15
""").fetchdf().to_string(index=False))

In [ ]:
display(con.execute("""
    SELECT
        COUNT(*) AS total_forecasts,

        COUNT(*) FILTER (
            WHERE d.date_key IS NULL
        ) AS unmatched_date_keys,

        COUNT(*) FILTER (
            WHERE d.date = d.month_end_date
        ) AS keys_on_month_end,

        COUNT(*) FILTER (
            WHERE d.date = d.month_start_date
        ) AS keys_on_month_start,

        COUNT(*) FILTER (
            WHERE d.date IS NOT NULL
              AND d.month_end_date IS NOT NULL
              AND d.date <> d.month_end_date
        ) AS keys_on_neither_month_boundary,

        COUNT(*) FILTER (
            WHERE DATE_TRUNC('month', f.forecast_month)::DATE
                  <> d.month_start_date
        ) AS forecast_month_start_mismatches

    FROM raw.fact_forecast f
    LEFT JOIN raw.dim_date d
        ON f.forecast_month_key = d.date_key
""").fetchdf().to_string(index=False))

In [ ]:
print("DATE DIMENSION COVERAGE")
display(con.execute("""
    SELECT
        MIN(date) AS earliest_date,
        MAX(date) AS latest_date,
        COUNT(*) AS total_dates,
        COUNT(*) FILTER (
            WHERE is_month_end = 'Y'
        ) AS month_end_dates
    FROM raw.dim_date
""").fetchdf().to_string(index=False))

print("\nUNMATCHED FORECAST MONTH KEYS")
display(con.execute("""
    SELECT
        f.forecast_id,
        f.forecast_month,
        f.forecast_month_key
    FROM raw.fact_forecast f
    LEFT JOIN raw.dim_date d
        ON f.forecast_month_key = d.date_key
    WHERE d.date_key IS NULL
    ORDER BY f.forecast_month, f.forecast_id
    LIMIT 10
""").fetchdf().to_string(index=False))

In [ ]:
display(con.execute("""
    SELECT
        f.forecast_month,
        f.forecast_month_key,
        COUNT(*) AS forecast_count
    FROM raw.fact_forecast f
    LEFT JOIN raw.dim_date d
        ON f.forecast_month_key = d.date_key
    WHERE d.date_key IS NULL
    GROUP BY
        f.forecast_month,
        f.forecast_month_key
    ORDER BY f.forecast_month
""").fetchdf().to_string(index=False))

In [ ]:
display(con.execute("""
    SELECT
        COUNT(*) AS total_rows,

        COUNT(DISTINCT forecast_id) AS unique_forecast_ids,

        COUNT(*) FILTER (
            WHERE forecast_id IS NULL
               OR NOT REGEXP_MATCHES(
                   forecast_id, '^FC-[0-9]{4}$'
               )
        ) AS invalid_forecast_ids,

        COUNT(*) FILTER (
            WHERE metric NOT IN (
                'headcount_end',
                'attrition_rate_monthly_pct'
            )
            OR metric IS NULL
        ) AS invalid_metrics,

        COUNT(*) FILTER (
            WHERE EXTRACT(DAY FROM forecast_month) <> 1
               OR forecast_month IS NULL
        ) AS invalid_forecast_months,

        COUNT(*) FILTER (
            WHERE forecast_value < 0
               OR forecast_value IS NULL
        ) AS invalid_forecast_values,

        COUNT(*) FILTER (
            WHERE lower_80 IS NULL
               OR upper_80 IS NULL
               OR lower_80 > forecast_value
               OR upper_80 < forecast_value
        ) AS invalid_intervals,

        COUNT(*) FILTER (
            WHERE trained_through IS NULL
               OR NOT REGEXP_MATCHES(
                   trained_through, '^[0-9]{4}-[0-9]{2}$'
               )
        ) AS invalid_trained_through

    FROM raw.fact_forecast
""").fetchdf().to_string(index=False))

In [ ]:
display(con.execute("""
    SELECT
        as_of_date,
        COUNT(*) AS record_count,
        COUNT(DISTINCT prediction_id) AS unique_prediction_ids,
        MIN(risk_score_12m) AS min_risk_score,
        MAX(risk_score_12m) AS max_risk_score
    FROM raw.fact_attrition_risk
    GROUP BY as_of_date
    ORDER BY as_of_date
""").fetchdf().to_string(index=False))

print("\nRECORDS WITH THE EARLIER AS-OF DATE")
display(con.execute("""
    SELECT
        prediction_id,
        employee_id,
        department_id,
        as_of_date,
        risk_score_12m,
        risk_band,
        model_version
    FROM raw.fact_attrition_risk
    WHERE as_of_date <> DATE '2026-09-30'
       OR as_of_date IS NULL
    ORDER BY prediction_id
""").fetchdf().to_string(index=False))

In [ ]:
display(con.execute("""
    SELECT
        a.prediction_id,
        a.employee_id,
        a.department_id,
        a.as_of_date,
        e.employee_key,
        e.valid_from,
        e.valid_to,
        e.employment_status,
        e.department_id AS employee_department_id,
        CASE
            WHEN e.employee_key IS NULL
                THEN 'NO_MATCHING_EMPLOYEE_VERSION'
            WHEN a.department_id <> e.department_id
                THEN 'DEPARTMENT_MISMATCH'
            WHEN e.valid_from > a.as_of_date
                OR (
                    e.valid_to IS NOT NULL
                    AND a.as_of_date >= e.valid_to
                )
                THEN 'OUTSIDE_EMPLOYEE_VALIDITY'
            ELSE 'MATCHED'
        END AS reference_check
    FROM raw.fact_attrition_risk a
    LEFT JOIN raw.dim_employee e
        ON a.employee_key = e.employee_key
       AND a.employee_id = e.employee_id
    WHERE a.as_of_date <> DATE '2026-09-30'
       OR a.as_of_date IS NULL
    ORDER BY a.prediction_id
""").fetchdf().to_string(index=False))

In [ ]:
display(con.execute("""
    SELECT
        a.prediction_id,
        a.employee_id,
        a.employee_key AS linked_employee_key,
        a.as_of_date,
        e.employee_key AS historical_employee_key,
        e.department_id AS historical_department_id,
        e.valid_from,
        e.valid_to,
        e.employment_status,
        e.is_current
    FROM raw.fact_attrition_risk a
    LEFT JOIN raw.dim_employee e
        ON a.employee_id = e.employee_id
       AND e.valid_from <= a.as_of_date
       AND (
           e.valid_to IS NULL
           OR a.as_of_date < e.valid_to
       )
    WHERE a.as_of_date <> DATE '2026-09-30'
       OR a.as_of_date IS NULL
    ORDER BY a.prediction_id
""").fetchdf().to_string(index=False))

In [ ]:
display(con.execute("""
    SELECT
        column_name,
        data_type,
        nullable,
        rule_or_allowed_values,
        description
    FROM raw.data_contracts
    WHERE feed = 'AI attrition risk (fact_attrition_risk)'
    ORDER BY column_name
""").fetchdf().to_string(index=False))

In [ ]:
display(con.execute("""
    SELECT
        as_of_date,
        risk_band,
        COUNT(*) AS record_count,
        MIN(risk_score_12m) AS min_score,
        MAX(risk_score_12m) AS max_score
    FROM raw.fact_attrition_risk
    GROUP BY as_of_date, risk_band
    ORDER BY as_of_date, min_score DESC
""").fetchdf().to_string(index=False))

In [ ]:
display(con.execute("""
    SELECT
        prediction_id,
        employee_id,
        as_of_date,
        risk_score_12m,
        risk_band
    FROM raw.fact_attrition_risk
    WHERE as_of_date = DATE '2026-09-30'
      AND risk_score_12m BETWEEN 0.125 AND 0.129
    ORDER BY risk_score_12m DESC, prediction_id
""").fetchdf().to_string(index=False))

In [ ]:
display(con.execute("""
    WITH score_groups AS (
        SELECT
            risk_score_12m,
            COUNT(*) AS records_at_score,
            COUNT(*) FILTER (
                WHERE risk_band = 'High'
            ) AS high_count,
            COUNT(*) FILTER (
                WHERE risk_band = 'Medium'
            ) AS medium_count,
            COUNT(*) FILTER (
                WHERE risk_band = 'Low'
            ) AS low_count
        FROM raw.fact_attrition_risk
        WHERE as_of_date = DATE '2026-09-30'
        GROUP BY risk_score_12m
    )
    SELECT *
    FROM score_groups
    WHERE records_at_score > 1
    ORDER BY risk_score_12m DESC
    LIMIT 20
""").fetchdf().to_string(index=False))

In [ ]:
display(con.execute("""
    SELECT
        COUNT(*) AS total_records,

        COUNT(*) FILTER (
            WHERE risk_band = 'High'
        ) AS high_records,

        ROUND(
            100.0 * COUNT(*) FILTER (
                WHERE risk_band = 'High'
            ) / COUNT(*),
            2
        ) AS high_percent,

        COUNT(*) FILTER (
            WHERE risk_band = 'Medium'
        ) AS medium_records,

        ROUND(
            100.0 * COUNT(*) FILTER (
                WHERE risk_band = 'Medium'
            ) / COUNT(*),
            2
        ) AS medium_percent,

        COUNT(*) FILTER (
            WHERE risk_band = 'Low'
        ) AS low_records,

        ROUND(
            100.0 * COUNT(*) FILTER (
                WHERE risk_band = 'Low'
            ) / COUNT(*),
            2
        ) AS low_percent

    FROM raw.fact_attrition_risk
    WHERE as_of_date = DATE '2026-09-30'
""").fetchdf().to_string(index=False))

In [ ]:
attrition_validation_status = {
    "feed": "fact_attrition_risk",
    "input_rows": 430,
    "unique_prediction_ids": 430,
    "missing_required_fields_in_checks": 0,
    "invalid_prediction_id_formats": 0,
    "invalid_risk_scores": 0,
    "invalid_risk_band_values": 0,
    "invalid_employee_or_department_ids": 0,
    "as_of_dates": {
        "2026-03-31": 5,
        "2026-09-30": 425,
    },
    "march_records_with_historical_employee_key_mismatch": 2,
    "september_band_counts": {
        "High": 43,
        "Medium": 86,
        "Low": 296,
    },
    "risk_band_tie_handling": (
        "Unresolved: score 0.127 appears in both Medium and Low"
    ),
    "raw_source_modified": False,
}

for key, value in attrition_validation_status.items():
    print(f"{key}: {value}")

In [ ]:
from pathlib import Path

notes_path = Path.cwd() / "docs" / "PQ-01_contract_notes.md"

attrition_notes = """

## Fact Attrition Risk — Validation Evidence

- Input records: 430; unique prediction IDs: 430.
- Missing required fields in the checks performed: 0.
- Invalid prediction ID formats: 0.
- Invalid risk scores outside 0–1: 0.
- Invalid risk-band values: 0.
- Unmatched employee or department IDs in the checks performed: 0.
- As-of-date distribution: 5 records dated 2026-03-31 and
  425 records dated 2026-09-30.
- Two March records reference an employee key that differs from
  the employee version valid on their as-of date.
- September risk-band counts: High 43 (10.12%), Medium 86 (20.24%),
  Low 296 (69.65%).
- The score 0.127 occurs in both Medium and Low bands. The
  percentile and tie-handling method remains unresolved.
- Raw source data was not intentionally modified.

### Outstanding Contract Questions

1. The contract requires the same as-of date for the whole batch.
   The five March records conflict with the 425 September records.
2. Two March records have employee keys inconsistent with the
   historical employee version valid on their as-of date.
3. The tie-handling rule for risk-band percentile assignment is
   not documented in the inspected sources.

These findings cover the checks listed above; they do not establish
that every attrition contract requirement has passed.
"""

if not notes_path.exists():
    raise FileNotFoundError(
        f"Expected project notes file not found: {notes_path}"
    )

with notes_path.open("a", encoding="utf-8") as f:
    f.write(attrition_notes)

print("Attrition validation evidence appended.")
print("Documentation path:", notes_path.resolve())
print("File exists:", notes_path.exists())

In [ ]:
display(con.execute("""
    SELECT
        COUNT(*) AS total_rows,

        COUNT(DISTINCT task_id) AS unique_task_ids,

        COUNT(*) FILTER (
            WHERE task_id IS NULL
               OR NOT REGEXP_MATCHES(
                   task_id, '^TASK-[0-9]{5}$'
               )
        ) AS invalid_task_ids,

        COUNT(*) FILTER (
            WHERE status NOT IN (
                'Done',
                'In Progress',
                'Blocked',
                'To Do',
                'Cancelled'
            )
            OR status IS NULL
        ) AS invalid_status_values,

        COUNT(*) FILTER (
            WHERE priority NOT IN ('P1', 'P2', 'P3', 'P4')
               OR priority IS NULL
        ) AS invalid_priorities,

        COUNT(*) FILTER (
            WHERE story_points NOT IN (1, 2, 3, 5, 8, 13)
               OR story_points IS NULL
        ) AS invalid_story_points,

        COUNT(*) FILTER (
            WHERE reopened_count < 0
               OR reopened_count IS NULL
        ) AS invalid_reopened_counts

    FROM raw.fact_task
""").fetchdf().to_string(index=False))

In [ ]:
display(con.execute("""
    SELECT
        COUNT(*) FILTER (
            WHERE created_date > started_date
        ) AS created_after_started,

        COUNT(*) FILTER (
            WHERE status <> 'To Do'
              AND started_date IS NULL
        ) AS missing_started_date_outside_todo,

        COUNT(*) FILTER (
            WHERE status = 'Done'
              AND completed_date IS NULL
        ) AS done_without_completed_date,

        COUNT(*) FILTER (
            WHERE completed_date < started_date
        ) AS completed_before_started,

        COUNT(*) FILTER (
            WHERE due_date < started_date
        ) AS due_before_started,

        COUNT(*) FILTER (
            WHERE assignee_employee_id NOT IN (
                SELECT employee_id FROM raw.dim_employee
            )
        ) AS unmatched_employee_ids,

        COUNT(*) FILTER (
            WHERE team_id NOT IN (
                SELECT team_id FROM raw.dim_team
            )
        ) AS unmatched_team_ids,

        COUNT(*) FILTER (
            WHERE sprint_id NOT IN (
                SELECT sprint_id FROM raw.dim_sprint
            )
        ) AS unmatched_sprint_ids

    FROM raw.fact_task
""").fetchdf().to_string(index=False))


In [ ]:
display(con.execute("""
    SELECT
        task_id,
        status,
        assignee_employee_id,
        created_date,
        started_date,
        completed_date,
        due_date,
        CASE
            WHEN status = 'Done'
                 AND completed_date IS NULL
                THEN 'MISSING_COMPLETED_DATE'
            WHEN completed_date < started_date
                THEN 'COMPLETED_BEFORE_STARTED'
        END AS violation_reason
    FROM raw.fact_task
    WHERE
        (status = 'Done' AND completed_date IS NULL)
        OR completed_date < started_date
    ORDER BY task_id
""").fetchdf().to_string(index=False))

In [ ]:
task_violation_summary = con.execute("""
    SELECT
        CASE
            WHEN status = 'Done'
                 AND completed_date IS NULL
                THEN 'MISSING_COMPLETED_DATE'
            WHEN completed_date < started_date
                THEN 'COMPLETED_BEFORE_STARTED'
        END AS quarantine_reason,
        COUNT(*) AS affected_records
    FROM raw.fact_task
    WHERE
        (status = 'Done' AND completed_date IS NULL)
        OR completed_date < started_date
    GROUP BY 1
    ORDER BY 1
""").fetchdf()

display(task_violation_summary)

print(
    "Total task records to quarantine:",
    int(task_violation_summary["affected_records"].sum())
)

In [ ]:
from pathlib import Path

project_root = Path.cwd()

for relative_path in [
    "src/etl_v2.py",
    "src/etl_multi.py",
    "docs/PQ-01_contract_notes.md",
    "outputs/pq01",
]:
    path = project_root / relative_path
    print(f"{relative_path}: {'EXISTS' if path.exists() else 'NOT FOUND'}")

if (project_root / "src/etl_v2.py").exists():
    print("\n--- Existing payroll ETL functions ---")
    print(
        (project_root / "src/etl_v2.py").read_text(encoding="utf-8")[:5000]
    )

In [ ]:
from pathlib import Path

project_root = Path.cwd()
etl_multi_path = project_root / "src" / "etl_multi.py"

etl_multi_code = r'''
import json
from pathlib import Path

import duckdb
import pandas as pd

ROOT = Path(__file__).resolve().parent.parent
DB_PATH = ROOT / "05_Starter" / "pulseiq.duckdb"
OUTPUT_DIR = ROOT / "outputs" / "pq01"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def read_raw_table(table_name):
    """Read a raw table without modifying the source database."""
    con = duckdb.connect(str(DB_PATH), read_only=True)
    try:
        return con.execute(
            f"SELECT * FROM raw.{table_name}"
        ).fetchdf()
    finally:
        con.close()


def add_reason(reasons, mask, reason_code):
    """Attach a reason code to every row selected by a boolean mask."""
    mask = mask.fillna(False)
    for idx in reasons.index[mask]:
        if reason_code not in reasons.at[idx]:
            reasons.at[idx].append(reason_code)


def split_passed_quarantine(df, reasons):
    """Split records while retaining every failed record and its reasons."""
    result = df.copy()
    result["quarantine_reason"] = reasons.apply(
        lambda values: "|".join(values) if values else None
    )

    quarantine = result[result["quarantine_reason"].notna()].copy()
    passed = result[result["quarantine_reason"].isna()].copy()

    if len(df) != len(passed) + len(quarantine):
        raise AssertionError("Input/pass/quarantine counts do not reconcile.")

    return passed, quarantine


def write_feed_outputs(table_name, passed, quarantine, input_count, warnings=None):
    """Write feed outputs and a reconciliation summary."""
    passed_path = OUTPUT_DIR / f"{table_name}_passed.csv"
    quarantine_path = OUTPUT_DIR / f"{table_name}_quarantine.csv"
    summary_path = OUTPUT_DIR / f"{table_name}_summary.json"

    passed.drop(columns=["quarantine_reason"], errors="ignore").to_csv(
        passed_path, index=False
    )
    quarantine.to_csv(quarantine_path, index=False)

    reason_counts = {}
    if not quarantine.empty:
        reason_counts = {
            str(reason): int(count)
            for reason, count in (
                quarantine["quarantine_reason"]
                .str.split("|")
                .explode()
                .value_counts()
                .items()
            )
        }

    summary = {
        "table": table_name,
        "rows_in": int(input_count),
        "rows_passed": int(len(passed)),
        "rows_quarantined": int(len(quarantine)),
        "reconciliation_ok": (
            int(input_count) == len(passed) + len(quarantine)
        ),
        "quarantine_reasons": reason_counts,
        "warnings": warnings or [],
    }

    if not summary["reconciliation_ok"]:
        raise AssertionError(f"Reconciliation failed for {table_name}")

    summary_path.write_text(
        json.dumps(summary, indent=2),
        encoding="utf-8"
    )

    return summary
'''

etl_multi_path.write_text(etl_multi_code, encoding="utf-8")

print("Created:", etl_multi_path)
print("File exists:", etl_multi_path.exists())
print("File size:", etl_multi_path.stat().st_size, "bytes")

In [ ]:
from pathlib import Path

project_root = Path.cwd()
etl_multi_path = project_root / "src" / "etl_multi.py"

etl_multi_code = r'''
import json
from pathlib import Path

import duckdb
import pandas as pd

ROOT = Path(__file__).resolve().parent.parent
DB_PATH = ROOT / "05_Starter" / "pulseiq.duckdb"
OUTPUT_DIR = ROOT / "outputs" / "pq01"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def read_raw_table(table_name):
    """Read a raw table without modifying the source database."""
    con = duckdb.connect(str(DB_PATH), read_only=True)
    try:
        return con.execute(
            f"SELECT * FROM raw.{table_name}"
        ).fetchdf()
    finally:
        con.close()


def add_reason(reasons, mask, reason_code):
    """Attach a reason code to every row selected by a boolean mask."""
    mask = mask.fillna(False)
    for idx in reasons.index[mask]:
        if reason_code not in reasons.at[idx]:
            reasons.at[idx].append(reason_code)


def split_passed_quarantine(df, reasons):
    """Split records while retaining every failed record and its reasons."""
    result = df.copy()
    result["quarantine_reason"] = reasons.apply(
        lambda values: "|".join(values) if values else None
    )

    quarantine = result[result["quarantine_reason"].notna()].copy()
    passed = result[result["quarantine_reason"].isna()].copy()

    if len(df) != len(passed) + len(quarantine):
        raise AssertionError("Input/pass/quarantine counts do not reconcile.")

    return passed, quarantine


def write_feed_outputs(table_name, passed, quarantine, input_count, warnings=None):
    """Write feed outputs and a reconciliation summary."""
    passed_path = OUTPUT_DIR / f"{table_name}_passed.csv"
    quarantine_path = OUTPUT_DIR / f"{table_name}_quarantine.csv"
    summary_path = OUTPUT_DIR / f"{table_name}_summary.json"

    passed.drop(columns=["quarantine_reason"], errors="ignore").to_csv(
        passed_path, index=False
    )
    quarantine.to_csv(quarantine_path, index=False)

    reason_counts = {}
    if not quarantine.empty:
        reason_counts = {
            str(reason): int(count)
            for reason, count in (
                quarantine["quarantine_reason"]
                .str.split("|")
                .explode()
                .value_counts()
                .items()
            )
        }

    summary = {
        "table": table_name,
        "rows_in": int(input_count),
        "rows_passed": int(len(passed)),
        "rows_quarantined": int(len(quarantine)),
        "reconciliation_ok": (
            int(input_count) == len(passed) + len(quarantine)
        ),
        "quarantine_reasons": reason_counts,
        "warnings": warnings or [],
    }

    if not summary["reconciliation_ok"]:
        raise AssertionError(f"Reconciliation failed for {table_name}")

    summary_path.write_text(
        json.dumps(summary, indent=2),
        encoding="utf-8"
    )

    return summary
'''

etl_multi_path.write_text(etl_multi_code, encoding="utf-8")

print("Created:", etl_multi_path)
print("File exists:", etl_multi_path.exists())
print("File size:", etl_multi_path.stat().st_size, "bytes")

In [ ]:
anomaly_function_code = r'''

def validate_anomaly():
    df = read_raw_table("fact_anomaly")
    reasons = pd.Series(
        [[] for _ in range(len(df))],
        index=df.index,
        dtype=object
    )

    required = [
        "result_id", "source_dataset", "record_id", "employee_id",
        "period", "anomaly_score", "flagged", "reason_code",
        "reason_text", "model_version", "scored_at"
    ]

    # Required fields must be present.
    for column in required:
        mask = df[column].isna()
        if column == "reason_text":
            mask = mask | df[column].astype("string").str.strip().eq("")
        add_reason(reasons, mask, f"MISSING_REQUIRED_{column.upper()}")

    # Unique, correctly formatted result IDs.
    add_reason(
        reasons,
        df["result_id"].isna()
        | ~df["result_id"].astype("string").str.match(
            r"^AN-[0-9]{4}$", na=False
        ),
        "INVALID_RESULT_ID"
    )
    add_reason(
        reasons,
        df["result_id"].duplicated(keep=False),
        "DUPLICATE_RESULT_ID"
    )

    # Contracted source datasets and value ranges.
    allowed_sources = {
        "attendance_monthly_features",
        "payroll_monthly_features",
    }
    add_reason(
        reasons,
        ~df["source_dataset"].isin(allowed_sources),
        "INVALID_SOURCE_DATASET"
    )
    add_reason(
        reasons,
        ~df["anomaly_score"].between(0, 1, inclusive="both"),
        "ANOMALY_SCORE_OUT_OF_RANGE"
    )
    add_reason(
        reasons,
        ~df["flagged"].isin(["Y", "N"]),
        "INVALID_FLAGGED_VALUE"
    )

    # Employee references must exist.
    employees = read_raw_table("dim_employee")
    valid_employee_ids = set(
        employees["employee_id"].dropna().astype(str)
    )
    add_reason(
        reasons,
        ~df["employee_id"].astype("string").isin(valid_employee_ids),
        "UNMATCHED_EMPLOYEE_ID"
    )

    # Money-at-risk semantics differ by source dataset.
    attendance_mask = df["source_dataset"].eq(
        "attendance_monthly_features"
    )
    payroll_mask = df["source_dataset"].eq(
        "payroll_monthly_features"
    )

    add_reason(
        reasons,
        attendance_mask & df["money_at_risk"].notna(),
        "UNEXPECTED_ATTENDANCE_MONEY_AT_RISK"
    )
    add_reason(
        reasons,
        payroll_mask & df["money_at_risk"].isna(),
        "MISSING_PAYROLL_MONEY_AT_RISK"
    )
    add_reason(
        reasons,
        payroll_mask & (df["money_at_risk"] < 0),
        "NEGATIVE_MONEY_AT_RISK"
    )

    passed, quarantine = split_passed_quarantine(df, reasons)

    warnings = [
        "No documented reason_code allowlist was found; values were not quarantined by reason code.",
        "scored_at is timezone-naive; UTC cannot be confirmed from the source schema.",
        "model_version naming convention is not fully specified.",
        "Attendance anomaly record IDs were validated through employee and month mapping rather than direct ID equality."
    ]

    return write_feed_outputs(
        "fact_anomaly",
        passed,
        quarantine,
        len(df),
        warnings
    )
'''

etl_multi_path = project_root / "src" / "etl_multi.py"
existing_code = etl_multi_path.read_text(encoding="utf-8")

if "def validate_anomaly():" in existing_code:
    print("validate_anomaly() already exists; no changes made.")
else:
    etl_multi_path.write_text(
        existing_code + anomaly_function_code,
        encoding="utf-8"
    )
    print("Added validate_anomaly() to:", etl_multi_path)

compile(etl_multi_path.read_text(encoding="utf-8"), str(etl_multi_path), "exec")
print("Syntax check: PASS")

In [ ]:
display(con.execute("""
    DESCRIBE raw.fact_attrition_risk
""").fetchdf())

display(con.execute("""
    DESCRIBE raw.dim_employee
""").fetchdf())

In [ ]:
attrition_function_code = r'''

def validate_attrition():
    df = read_raw_table("fact_attrition_risk")
    employees = read_raw_table("dim_employee")
    departments = read_raw_table("dim_department")

    reasons = pd.Series(
        [[] for _ in range(len(df))],
        index=df.index,
        dtype=object
    )

    # Required contract fields.
    required = [
        "prediction_id", "employee_id", "department_id",
        "as_of_date", "risk_score_12m", "risk_band",
        "model_version"
    ]
    for column in required:
        add_reason(
            reasons,
            df[column].isna(),
            f"MISSING_REQUIRED_{column.upper()}"
        )

    # Prediction ID format and uniqueness.
    add_reason(
        reasons,
        df["prediction_id"].isna()
        | ~df["prediction_id"].astype("string").str.match(
            r"^AR-[0-9]{4}$", na=False
        ),
        "INVALID_PREDICTION_ID"
    )
    add_reason(
        reasons,
        df["prediction_id"].duplicated(keep=False),
        "DUPLICATE_PREDICTION_ID"
    )

    # Score range and permitted risk-band values.
    add_reason(
        reasons,
        ~df["risk_score_12m"].between(0, 1, inclusive="both"),
        "RISK_SCORE_OUT_OF_RANGE"
    )
    add_reason(
        reasons,
        ~df["risk_band"].isin(["Low", "Medium", "High"]),
        "INVALID_RISK_BAND"
    )

    # Department reference.
    valid_departments = set(
        departments["department_id"].dropna().astype(str)
    )
    add_reason(
        reasons,
        ~df["department_id"].astype("string").isin(valid_departments),
        "UNMATCHED_DEPARTMENT_ID"
    )

    # Validate employee history using a half-open validity interval:
    # valid_from <= as_of_date < valid_to, or no valid_to.
    employee_history = employees[
        [
            "employee_key", "employee_id", "department_id",
            "employment_status", "exit_date", "valid_from", "valid_to"
        ]
    ].copy()

    employee_history["employee_id"] = (
        employee_history["employee_id"].astype("string")
    )

    valid_history_counts = []
    expected_keys = []
    expected_departments = []
    active_history_flags = []

    for _, row in df.iterrows():
        candidates = employee_history[
            employee_history["employee_id"].eq(str(row["employee_id"]))
        ]

        if pd.isna(row["as_of_date"]):
            candidates = candidates.iloc[0:0]
        else:
            candidates = candidates[
                (candidates["valid_from"] <= row["as_of_date"])
                & (
                    candidates["valid_to"].isna()
                    | (row["as_of_date"] < candidates["valid_to"])
                )
            ]

        valid_history_counts.append(len(candidates))

        if len(candidates) == 1:
            hist = candidates.iloc[0]
            expected_keys.append(hist["employee_key"])
            expected_departments.append(hist["department_id"])

            status = str(hist["employment_status"]).strip().lower()
            exit_date = hist["exit_date"]

            is_active = (
                status in {"active", "employed"}
                and (pd.isna(exit_date) or row["as_of_date"] <= exit_date)
            )
            active_history_flags.append(is_active)
        else:
            expected_keys.append(None)
            expected_departments.append(None)
            active_history_flags.append(False)

    history_count = pd.Series(valid_history_counts, index=df.index)
    expected_key = pd.Series(expected_keys, index=df.index)
    expected_department = pd.Series(expected_departments, index=df.index)
    active_history = pd.Series(active_history_flags, index=df.index)

    add_reason(
        reasons,
        history_count.eq(0),
        "NO_EMPLOYEE_VERSION_AS_OF_DATE"
    )
    add_reason(
        reasons,
        history_count.gt(1),
        "MULTIPLE_EMPLOYEE_VERSIONS_AS_OF_DATE"
    )
    add_reason(
        reasons,
        history_count.eq(1) & ~active_history,
        "EMPLOYEE_NOT_ACTIVE_AS_OF_DATE"
    )
    add_reason(
        reasons,
        history_count.eq(1)
        & df["employee_key"].notna()
        & df["employee_key"].ne(expected_key),
        "EMPLOYEE_KEY_NOT_VALID_AS_OF_DATE"
    )
    add_reason(
        reasons,
        history_count.eq(1)
        & df["department_id"].ne(expected_department),
        "DEPARTMENT_MISMATCH_AS_OF_DATE"
    )

    # Contract says one as_of_date per batch. Select the modal date
    # dynamically, and stop safely if the modal date is tied.
    date_counts = df["as_of_date"].value_counts(dropna=True)

    if len(date_counts) > 1:
        top_count = date_counts.max()
        modal_dates = date_counts[date_counts.eq(top_count)].index.tolist()

        if len(modal_dates) != 1:
            raise ValueError(
                "Multiple as_of_date values tie for the largest batch; "
                "cannot choose a reference date safely."
            )

        reference_date = modal_dates[0]
        add_reason(
            reasons,
            df["as_of_date"].notna()
            & df["as_of_date"].ne(reference_date),
            "INCONSISTENT_AS_OF_DATE"
        )

    passed, quarantine = split_passed_quarantine(df, reasons)

    warnings = [
        "Risk-band percentile and tie-handling rules are not fully specified; no percentile-based quarantine was applied.",
        "The modal as_of_date is used as a data-driven batch reference because the contract requires one date per batch."
    ]

    return write_feed_outputs(
        "fact_attrition_risk",
        passed,
        quarantine,
        len(df),
        warnings
    )
'''

existing_code = etl_multi_path.read_text(encoding="utf-8")

if "def validate_attrition():" in existing_code:
    print("validate_attrition() already exists; no changes made.")
else:
    etl_multi_path.write_text(
        existing_code + attrition_function_code,
        encoding="utf-8"
    )
    print("Added validate_attrition()")

compile(
    etl_multi_path.read_text(encoding="utf-8"),
    str(etl_multi_path),
    "exec"
)
print("Syntax check: PASS")

In [ ]:
forecast_function_code = r'''

def validate_forecast():
    df = read_raw_table("fact_forecast")
    departments = read_raw_table("dim_department")

    reasons = pd.Series(
        [[] for _ in range(len(df))],
        index=df.index,
        dtype=object
    )

    required = [
        "forecast_id", "metric", "department_id", "forecast_month",
        "forecast_value", "lower_80", "upper_80", "method",
        "trained_through"
    ]

    for column in required:
        add_reason(
            reasons,
            df[column].isna(),
            f"MISSING_REQUIRED_{column.upper()}"
        )

    # Unique forecast IDs and documented ID format.
    add_reason(
        reasons,
        df["forecast_id"].isna()
        | ~df["forecast_id"].astype("string").str.match(
            r"^FC-[0-9]{4}$", na=False
        ),
        "INVALID_FORECAST_ID"
    )
    add_reason(
        reasons,
        df["forecast_id"].duplicated(keep=False),
        "DUPLICATE_FORECAST_ID"
    )

    # Permitted forecast metrics.
    allowed_metrics = {
        "headcount_end",
        "attrition_rate_monthly_pct",
    }
    add_reason(
        reasons,
        ~df["metric"].isin(allowed_metrics),
        "INVALID_FORECAST_METRIC"
    )

    # Department must exist, unless the contract permits ALL.
    valid_departments = set(
        departments["department_id"].dropna().astype(str)
    )
    valid_departments.add("ALL")

    add_reason(
        reasons,
        ~df["department_id"].astype("string").isin(valid_departments),
        "INVALID_FORECAST_DEPARTMENT"
    )

    # Forecast month must be the first calendar day of its month.
    month = pd.to_datetime(df["forecast_month"], errors="coerce")
    add_reason(
        reasons,
        month.isna()
        | month.dt.day.ne(1),
        "FORECAST_MONTH_NOT_MONTH_START"
    )

    # Forecast values must be nonnegative.
    add_reason(
        reasons,
        df["forecast_value"].isna()
        | (df["forecast_value"] < 0),
        "NEGATIVE_OR_MISSING_FORECAST_VALUE"
    )

    # The 80% interval must contain the forecast value.
    add_reason(
        reasons,
        df["lower_80"].isna()
        | df["upper_80"].isna()
        | (df["lower_80"] > df["forecast_value"])
        | (df["upper_80"] < df["forecast_value"]),
        "INVALID_FORECAST_INTERVAL"
    )

    # trained_through must follow YYYY-MM format.
    trained = df["trained_through"].astype("string")
    valid_trained_format = trained.str.match(
        r"^[0-9]{4}-(0[1-9]|1[0-2])$", na=False
    )
    add_reason(
        reasons,
        ~valid_trained_format,
        "INVALID_TRAINED_THROUGH_FORMAT"
    )

    passed, quarantine = split_passed_quarantine(df, reasons)

    warnings = [
        "forecast_month_key uses month-end date keys while forecast_month is month-start.",
        "Forecasts extend through March 2027, beyond the current dim_date coverage; unmatched future date keys are not quarantined by this validator."
    ]

    return write_feed_outputs(
        "fact_forecast",
        passed,
        quarantine,
        len(df),
        warnings
    )
'''

existing_code = etl_multi_path.read_text(encoding="utf-8")

if "def validate_forecast():" in existing_code:
    print("validate_forecast() already exists; no changes made.")
else:
    etl_multi_path.write_text(
        existing_code + forecast_function_code,
        encoding="utf-8"
    )
    print("Added validate_forecast()")

compile(
    etl_multi_path.read_text(encoding="utf-8"),
    str(etl_multi_path),
    "exec"
)
print("Syntax check: PASS")

In [ ]:
task_function_code = r'''

def validate_task():
    df = read_raw_table("fact_task")
    employees = read_raw_table("dim_employee")
    teams = read_raw_table("dim_team")
    sprints = read_raw_table("dim_sprint")

    reasons = pd.Series(
        [[] for _ in range(len(df))],
        index=df.index,
        dtype=object
    )

    required = [
        "task_id", "team_id", "sprint_id", "assignee_employee_id",
        "task_type", "priority", "story_points", "created_date",
        "due_date", "status", "reopened_count"
    ]

    for column in required:
        add_reason(
            reasons,
            df[column].isna(),
            f"MISSING_REQUIRED_{column.upper()}"
        )

    add_reason(
        reasons,
        df["task_id"].isna()
        | ~df["task_id"].astype("string").str.match(
            r"^TASK-[0-9]{5}$", na=False
        ),
        "INVALID_TASK_ID"
    )
    add_reason(
        reasons,
        df["task_id"].duplicated(keep=False),
        "DUPLICATE_TASK_ID"
    )

    add_reason(
        reasons,
        ~df["status"].isin(
            ["Done", "In Progress", "Blocked", "To Do", "Cancelled"]
        ),
        "INVALID_STATUS"
    )
    add_reason(
        reasons,
        ~df["priority"].isin(["P1", "P2", "P3", "P4"]),
        "INVALID_PRIORITY"
    )
    add_reason(
        reasons,
        ~df["story_points"].isin([1, 2, 3, 5, 8, 13]),
        "INVALID_STORY_POINTS"
    )
    add_reason(
        reasons,
        ~df["task_type"].isin(["Feature", "Bug", "Chore", "Spike"]),
        "INVALID_TASK_TYPE"
    )
    add_reason(
        reasons,
        df["reopened_count"].isna() | (df["reopened_count"] < 0),
        "INVALID_REOPENED_COUNT"
    )

    # Referential integrity.
    valid_employee_ids = set(
        employees["employee_id"].dropna().astype(str)
    )
    valid_team_ids = set(teams["team_id"].dropna().astype(str))
    valid_sprint_ids = set(sprints["sprint_id"].dropna().astype(str))

    add_reason(
        reasons,
        ~df["assignee_employee_id"].astype("string").isin(
            valid_employee_ids
        ),
        "UNMATCHED_EMPLOYEE_ID"
    )
    add_reason(
        reasons,
        ~df["team_id"].astype("string").isin(valid_team_ids),
        "UNMATCHED_TEAM_ID"
    )
    add_reason(
        reasons,
        ~df["sprint_id"].astype("string").isin(valid_sprint_ids),
        "UNMATCHED_SPRINT_ID"
    )

    # Date consistency.
    add_reason(
        reasons,
        df["created_date"].notna()
        & df["started_date"].notna()
        & (df["created_date"] > df["started_date"]),
        "CREATED_AFTER_STARTED"
    )
    add_reason(
        reasons,
        df["started_date"].isna()
        & df["status"].ne("To Do"),
        "MISSING_STARTED_DATE"
    )
    add_reason(
        reasons,
        df["status"].eq("Done") & df["completed_date"].isna(),
        "MISSING_COMPLETED_DATE"
    )
    add_reason(
        reasons,
        df["completed_date"].notna()
        & df["started_date"].notna()
        & (df["completed_date"] < df["started_date"]),
        "COMPLETED_BEFORE_STARTED"
    )
    add_reason(
        reasons,
        df["due_date"].notna()
        & df["started_date"].notna()
        & (df["due_date"] < df["started_date"]),
        "DUE_BEFORE_STARTED"
    )

    # Due date must match the sprint end date.
    sprint_ends = sprints[["sprint_id", "end_date"]].drop_duplicates(
        subset=["sprint_id"]
    )
    sprint_end_map = sprint_ends.set_index("sprint_id")["end_date"]
    expected_end = df["sprint_id"].map(sprint_end_map)

    add_reason(
        reasons,
        expected_end.isna()
        | df["due_date"].ne(expected_end),
        "DUE_DATE_MISMATCH_SPRINT_END"
    )

    passed, quarantine = split_passed_quarantine(df, reasons)

    warnings = [
        "The contract requires started_date to be blank only for To Do; other status-specific completion rules were not inferred beyond the documented Done rule."
    ]

    return write_feed_outputs(
        "fact_task",
        passed,
        quarantine,
        len(df),
        warnings
    )
'''

existing_code = etl_multi_path.read_text(encoding="utf-8")

if "def validate_task():" in existing_code:
    print("validate_task() already exists; no changes made.")
else:
    etl_multi_path.write_text(
        existing_code + task_function_code,
        encoding="utf-8"
    )
    print("Added validate_task()")

compile(
    etl_multi_path.read_text(encoding="utf-8"),
    str(etl_multi_path),
    "exec"
)
print("Syntax check: PASS")

In [ ]:
import importlib.util
from pathlib import Path
import json

etl_multi_path = Path.cwd() / "src" / "etl_multi.py"

spec = importlib.util.spec_from_file_location(
    "etl_multi",
    etl_multi_path
)
etl_multi = importlib.util.module_from_spec(spec)
spec.loader.exec_module(etl_multi)

anomaly_summary = etl_multi.validate_anomaly()

print("\nAnomaly validation summary:")
print(json.dumps(anomaly_summary, indent=2))

In [ ]:
attrition_summary = etl_multi.validate_attrition()

print("\nAttrition validation summary:")
print(json.dumps(attrition_summary, indent=2))

In [ ]:
forecast_summary = etl_multi.validate_forecast()

print("\nForecast validation summary:")
print(json.dumps(forecast_summary, indent=2))

In [ ]:
task_summary = etl_multi.validate_task()

print("\nTask validation summary:")
print(json.dumps(task_summary, indent=2))

In [ ]:
import json
from pathlib import Path

etl_multi_path = Path.cwd() / "src" / "etl_multi.py"

orchestrator_code = r'''

def run_all_feeds():
    """Run payroll and all four contracted feed validators."""
    import importlib.util

    # Reuse the existing, tested payroll ETL.
    payroll_path = ROOT / "src" / "etl_v2.py"
    payroll_spec = importlib.util.spec_from_file_location(
        "etl_v2_for_multi", payroll_path
    )
    payroll_module = importlib.util.module_from_spec(payroll_spec)
    payroll_spec.loader.exec_module(payroll_module)
    payroll_module.run_payroll_etl()

    payroll_summary_path = OUTPUT_DIR / "fact_payroll_summary.json"
    payroll_summary = json.loads(
        payroll_summary_path.read_text(encoding="utf-8")
    )

    summaries = [payroll_summary]
    summaries.append(validate_anomaly())
    summaries.append(validate_attrition())
    summaries.append(validate_forecast())
    summaries.append(validate_task())

    total_in = sum(item["rows_in"] for item in summaries)
    total_passed = sum(item["rows_passed"] for item in summaries)
    total_quarantined = sum(
        item["rows_quarantined"] for item in summaries
    )

    overall = {
        "pipeline": "PQ-01 multi-feed ETL",
        "feeds": summaries,
        "total_rows_in": total_in,
        "total_rows_passed": total_passed,
        "total_rows_quarantined": total_quarantined,
        "all_feeds_reconciled": all(
            item["reconciliation_ok"] for item in summaries
        ),
        "overall_reconciliation_ok": (
            total_in == total_passed + total_quarantined
        ),
    }

    summary_path = OUTPUT_DIR / "pq01_overall_summary.json"
    summary_path.write_text(
        json.dumps(overall, indent=2),
        encoding="utf-8"
    )

    print(json.dumps({
        "pipeline": overall["pipeline"],
        "feed_count": len(summaries),
        "total_rows_in": total_in,
        "total_rows_passed": total_passed,
        "total_rows_quarantined": total_quarantined,
        "all_feeds_reconciled": overall["all_feeds_reconciled"],
        "overall_reconciliation_ok": overall["overall_reconciliation_ok"],
        "summary_file": str(summary_path),
    }, indent=2))

    if not overall["all_feeds_reconciled"]:
        raise AssertionError("At least one feed failed reconciliation.")
    if not overall["overall_reconciliation_ok"]:
        raise AssertionError("Overall pipeline reconciliation failed.")

    return overall
'''

existing_code = etl_multi_path.read_text(encoding="utf-8")

if "def run_all_feeds():" in existing_code:
    print("run_all_feeds() already exists; no changes made.")
else:
    etl_multi_path.write_text(
        existing_code + orchestrator_code,
        encoding="utf-8"
    )
    print("Added run_all_feeds()")

compile(
    etl_multi_path.read_text(encoding="utf-8"),
    str(etl_multi_path),
    "exec"
)
print("Syntax check: PASS")

In [ ]:
overall_summary = etl_multi.run_all_feeds()

print("\nFeed-by-feed reconciliation:")
for feed in overall_summary["feeds"]:
    print(
        f'{feed["table"]}: '
        f'in={feed["rows_in"]}, '
        f'passed={feed["rows_passed"]}, '
        f'quarantined={feed["rows_quarantined"]}, '
        f'reconciled={feed["reconciliation_ok"]}'
    )

In [ ]:
import importlib
import etl_multi

etl_multi = importlib.reload(etl_multi)

print("Module reloaded successfully.")
print("run_all_feeds exists:", hasattr(etl_multi, "run_all_feeds"))

In [ ]:
import importlib.util
from pathlib import Path

etl_multi_path = Path.cwd() / "src" / "etl_multi.py"

print("File exists:", etl_multi_path.exists())

spec = importlib.util.spec_from_file_location(
    "etl_multi",
    etl_multi_path
)

etl_multi = importlib.util.module_from_spec(spec)
spec.loader.exec_module(etl_multi)

print("Module loaded successfully.")
print("run_all_feeds exists:", hasattr(etl_multi, "run_all_feeds"))

In [ ]:
overall_summary = etl_multi.run_all_feeds()

print("\nFeed-by-feed reconciliation:")
for feed in overall_summary["feeds"]:
    print(
        f'{feed["table"]}: '
        f'in={feed["rows_in"]}, '
        f'passed={feed["rows_passed"]}, '
        f'quarantined={feed["rows_quarantined"]}, '
        f'reconciled={feed["reconciliation_ok"]}'
    )

In [ ]:
import hashlib
from pathlib import Path

output_dir = Path.cwd() / "outputs" / "pq01"

def calculate_hashes(folder):
    hashes = {}
    for path in sorted(folder.iterdir()):
        if path.is_file():
            hashes[path.name] = hashlib.sha256(
                path.read_bytes()
            ).hexdigest()
    return hashes

# Capture outputs from the first run.
hashes_before = calculate_hashes(output_dir)

# Execute the complete pipeline a second time.
print("Running pipeline for repeatability test...")
overall_summary_second = etl_multi.run_all_feeds()

# Capture outputs from the second run.
hashes_after = calculate_hashes(output_dir)

# Compare every output file.
all_files = sorted(set(hashes_before) | set(hashes_after))
comparison = []

for filename in all_files:
    comparison.append({
        "file": filename,
        "identical": (
            hashes_before.get(filename) == hashes_after.get(filename)
        )
    })

import pandas as pd
comparison_df = pd.DataFrame(comparison)
display(comparison_df)

print("\nFiles checked:", len(comparison_df))
print("All outputs identical:", bool(comparison_df["identical"].all()))

In [ ]:
import duckdb
from pathlib import Path
import pandas as pd

db_path = Path.cwd() / "05_Starter" / "pulseiq.duckdb"

con_check = duckdb.connect(str(db_path), read_only=True)

try:
    source_counts = con_check.execute("""
        SELECT 'fact_payroll' AS table_name, COUNT(*) AS row_count
        FROM raw.fact_payroll

        UNION ALL

        SELECT 'fact_anomaly', COUNT(*)
        FROM raw.fact_anomaly

        UNION ALL

        SELECT 'fact_attrition_risk', COUNT(*)
        FROM raw.fact_attrition_risk

        UNION ALL

        SELECT 'fact_forecast', COUNT(*)
        FROM raw.fact_forecast

        UNION ALL

        SELECT 'fact_task', COUNT(*)
        FROM raw.fact_task

        UNION ALL

        SELECT 'dim_employee', COUNT(*)
        FROM raw.dim_employee
    """).fetchdf()
finally:
    con_check.close()

display(source_counts)

In [ ]:
import subprocess
from pathlib import Path

project_root = Path.cwd()

result = subprocess.run(
    ["git", "status", "--short"],
    cwd=project_root,
    capture_output=True,
    text=True
)

print("Git status exit code:", result.returncode)
print(result.stdout if result.stdout.strip() else "No changes detected.")

if result.stderr.strip():
    print("\nGit messages:")
    print(result.stderr)

In [ ]:
from pathlib import Path

root = Path.cwd()

gitignore_path = root / ".gitignore"
print(".gitignore exists:", gitignore_path.exists())

if gitignore_path.exists():
    print("\n--- Current .gitignore ---")
    print(gitignore_path.read_text(encoding="utf-8"))

for folder_name in ["src", "docs", "outputs", "04_Validation_Vault"]:
    folder = root / folder_name
    print(f"\n--- {folder_name} ---")

    if folder.exists():
        for path in sorted(folder.rglob("*")):
            if path.is_file():
                print(f"{path.relative_to(root)} ({path.stat().st_size:,} bytes)")

In [ ]:
from pathlib import Path

root = Path.cwd()
gitignore_path = root / ".gitignore"

gitignore_content = """
# Python cache and compiled files
__pycache__/
*.py[cod]

# Jupyter temporary files
.ipynb_checkpoints/

# Virtual environments and local secrets
.venv/
venv/
.env
.env.*

# Source datasets and validation answer materials
03_Data/
04_Validation_Vault/

# Local DuckDB database files
05_Starter/*.duckdb
05_Starter/*.duckdb.wal
05_Starter/*.duckdb.tmp

# Generated passed datasets (retain quarantine evidence and JSON summaries)
outputs/pq01/*_passed.csv
"""

if gitignore_path.exists():
    print("A .gitignore already exists. No changes made.")
else:
    gitignore_path.write_text(
        gitignore_content.strip() + "\n",
        encoding="utf-8"
    )
    print("Created:", gitignore_path)

print("\n--- .gitignore contents ---")
print(gitignore_path.read_text(encoding="utf-8"))

In [ ]:
from pathlib import Path

root = Path.cwd()

folders_to_check = [
    root / "05_Starter",
    root / "07_Working_Templates",
]

for folder in folders_to_check:
    print(f"\n{'=' * 60}")
    print(f"Folder: {folder}")
    print(f"Exists: {folder.exists()}")

    if folder.exists():
        for item in sorted(folder.iterdir()):
            kind = "DIR " if item.is_dir() else "FILE"
            size = item.stat().st_size if item.is_file() else "-"
            print(f"{kind} | {item.name} | {size} bytes")

In [ ]:
from pathlib import Path
import csv

root = Path.cwd()

# 1. Inspect demo_users.csv without exposing full row contents
csv_path = root / "05_Starter" / "demo_users.csv"

print("DEMO CSV CHECK")
with csv_path.open("r", encoding="utf-8-sig", newline="") as f:
    reader = csv.DictReader(f)
    rows = list(reader)
    print("Columns:", reader.fieldnames)
    print("Row count:", len(rows))
    print("Potentially sensitive column names:",
          [c for c in (reader.fieldnames or [])
           if any(word in c.lower() for word in
                  ["password", "secret", "token", "email", "phone", "key"])])

# 2. Inspect the six working templates
template_dir = root / "07_Working_Templates"

for path in sorted(template_dir.glob("*.md")):
    print(f"\n{'=' * 60}")
    print(f"FILE: {path.name}")
    print(path.read_text(encoding="utf-8"))

# 3. Check whether the database is ignored by Git
print(f"\n{'=' * 60}")
print("DATABASE EXISTS:", (root / "05_Starter" / "pulseiq.duckdb").exists())

In [ ]:
import subprocess
from pathlib import Path

root = Path.cwd()

checks = [
    ".gitignore",
    "05_Starter/pulseiq.duckdb",
    "03_Data/",
    "04_Validation_Vault/",
    "outputs/pq01/payroll_passed.csv",
]

for item in checks:
    result = subprocess.run(
        ["git", "check-ignore", "-v", item],
        cwd=root,
        capture_output=True,
        text=True
    )

    print(f"\n{item}")
    if result.returncode == 0:
        print("IGNORED:", result.stdout.strip())
    else:
        print("NOT IGNORED (or not matched):", result.stderr.strip() or "No matching ignore rule")

print("\n--- Current Git status ---")
result = subprocess.run(
    ["git", "status", "--short"],
    cwd=root,
    capture_output=True,
    text=True,
    check=True
)
print(result.stdout)

In [ ]:
import subprocess
from pathlib import Path

root = Path.cwd()

# Preview files Git would consider, including ignored files for review
result = subprocess.run(
    ["git", "ls-files", "--others", "--exclude-standard"],
    cwd=root,
    capture_output=True,
    text=True,
    check=True
)

candidates = sorted(result.stdout.splitlines())

print("FILES ELIGIBLE FOR STAGING")
for path in candidates:
    print(path)

print("\n--- Safety checks ---")
blocked = [
    p for p in candidates
    if p.startswith(("03_Data/", "04_Validation_Vault/"))
    or p.endswith((".duckdb", ".duckdb.wal", ".duckdb.tmp"))
    or (p.startswith("outputs/pq01/") and p.endswith("_passed.csv"))
]

print("Excluded files appearing in candidate list:", blocked)
print("Candidate file count:", len(candidates))

In [ ]:
from pathlib import Path
import csv
import subprocess

root = Path.cwd()

# 1. Review the original ETL prototype before deciding whether to stage it
prototype = root / "src" / "etl.py"
print("=== src/etl.py ===")
print(prototype.read_text(encoding="utf-8"))

# 2. Inspect quarantine CSV structure and basic risk indicators
print("\n=== QUARANTINE FILE REVIEW ===")
for path in sorted((root / "outputs" / "pq01").glob("*_quarantine.csv")):
    with path.open("r", encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)
        rows = list(reader)
        columns = reader.fieldnames or []

    sensitive_columns = [
        c for c in columns
        if any(word in c.lower() for word in
               ["password", "secret", "token", "email", "phone", "address"])
    ]

    print(f"\n{path.name}")
    print("Rows:", len(rows))
    print("Columns:", columns)
    print("Sensitive-looking columns:", sensitive_columns)

# 3. Check whether the prototype contains hardcoded test IDs
print("\n=== PROTOTYPE SAFETY CHECK ===")
source = prototype.read_text(encoding="utf-8")
for marker in ["PI-X0001", "PI-X0002", "PI-X0003", "PI-X0004"]:
    print(f"{marker} present:", marker in source)

# 4. Confirm no protected file has already been staged
print("\n=== CURRENT STAGED FILES ===")
result = subprocess.run(
    ["git", "diff", "--cached", "--name-only"],
    cwd=root,
    capture_output=True,
    text=True,
    check=True
)
print(result.stdout or "(Nothing staged yet)")

In [ ]:
from pathlib import Path

root = Path.cwd()
gitignore_path = root / ".gitignore"

extra_rules = """
# Keep row-level quarantine data local; commit aggregate JSON evidence
outputs/pq01/*_quarantine.csv

# Original hardcoded prototype; use etl_v2.py and etl_multi.py
src/etl.py
"""

current = gitignore_path.read_text(encoding="utf-8")

if "outputs/pq01/*_quarantine.csv" not in current:
    gitignore_path.write_text(
        current.rstrip() + "\n\n" + extra_rules.strip() + "\n",
        encoding="utf-8"
    )
    print("Updated .gitignore with quarantine and prototype exclusions.")
else:
    print("Quarantine exclusion already exists; no changes made.")

print("\n--- Updated .gitignore ---")
print(gitignore_path.read_text(encoding="utf-8"))

In [ ]:
import subprocess
from pathlib import Path

root = Path.cwd()

paths_to_stage = [
    ".gitignore",
    "05_Starter/demo_users.csv",
    "05_Starter/load_to_duckdb.py",
    "05_Starter/metric_views_examples.sql",
    "05_Starter/requirements.txt",
    "07_Working_Templates/",
    "PQ-01_ETL_Validation.ipynb",
    "docs/",
    "src/etl_v2.py",
    "src/etl_multi.py",
    "outputs/pq01/",
]

result = subprocess.run(
    ["git", "add", "--"] + paths_to_stage,
    cwd=root,
    capture_output=True,
    text=True
)

if result.returncode != 0:
    print("STAGING FAILED")
    print(result.stderr)
else:
    print("Staging command completed.")

print("\n--- Staged files ---")
result = subprocess.run(
    ["git", "diff", "--cached", "--name-only"],
    cwd=root,
    capture_output=True,
    text=True,
    check=True
)
print(result.stdout)

print("\n--- Staged summary ---")
result = subprocess.run(
    ["git", "diff", "--cached", "--stat"],
    cwd=root,
    capture_output=True,
    text=True,
    check=True
)
print(result.stdout)

In [188]:
import subprocess
import json
import re
from pathlib import Path

root = Path.cwd()

def git_output(args):
    result = subprocess.run(
        ["git"] + args,
        cwd=root,
        capture_output=True,
        text=True,
        check=True
    )
    return result.stdout

staged = git_output(["diff", "--cached", "--name-only"]).splitlines()

# 1. Git whitespace/error check
print("=== GIT WHITESPACE CHECK ===")
check = subprocess.run(
    ["git", "diff", "--cached", "--check"],
    cwd=root,
    capture_output=True,
    text=True
)
print("PASS" if check.returncode == 0 and not check.stdout and not check.stderr
      else (check.stdout + check.stderr or "Review required"))

# 2. Validate summary JSON totals
print("\n=== SUMMARY RECONCILIATION ===")
overall_path = root / "outputs/pq01/pq01_overall_summary.json"
overall = json.loads(overall_path.read_text(encoding="utf-8"))

print(json.dumps(overall, indent=2))

# 3. Check staged paths for protected files
print("\n=== PROTECTED FILE CHECK ===")
blocked = [
    p for p in staged
    if p.startswith(("03_Data/", "04_Validation_Vault/"))
    or p.endswith((".duckdb", ".duckdb.wal", ".duckdb.tmp"))
    or p.endswith("_passed.csv")
    or p.endswith("_quarantine.csv")
    or p == "src/etl.py"
]
print("PASS: no protected files staged" if not blocked else "REVIEW:", blocked)

# 4. Lightweight scan for common secret patterns in staged text files
print("\n=== SECRET PATTERN SCAN ===")
patterns = {
    "private key header": re.compile(r"-----BEGIN (?:RSA |EC |OPENSSH )?PRIVATE KEY-----"),
    "API key assignment": re.compile(r"(?i)\b(?:api[_-]?key|secret[_-]?key|access[_-]?token)\s*[:=]\s*['\"]?[^'\"\s]{12,}"),
    "password assignment": re.compile(r"(?i)\bpassword\s*[:=]\s*['\"][^'\"]{4,}['\"]"),
}
hits = []

for rel in staged:
    path = root / rel
    if not path.is_file():
        continue
    try:
        content = path.read_text(encoding="utf-8-sig")
    except (UnicodeDecodeError, OSError):
        continue

    for label, pattern in patterns.items():
        if pattern.search(content):
            hits.append((rel, label))

print("No common secret patterns detected." if not hits else "Review these matches:")
for hit in hits:
    print("-", hit[0], "|", hit[1])

# 5. Check notebook output for potentially exposed row-level identifiers
print("\n=== NOTEBOOK OUTPUT REVIEW ===")
notebook = json.loads((root / "PQ-01_ETL_Validation.ipynb").read_text(encoding="utf-8"))
output_cells = 0
employee_id_hits = 0

for cell in notebook.get("cells", []):
    if cell.get("cell_type") != "code":
        continue
    for output in cell.get("outputs", []):
        text = output.get("text", "")
        if isinstance(text, list):
            text = "".join(text)
        data = output.get("data", {})
        text += "".join(data.get("text/plain", [])) if isinstance(data.get("text/plain", []), list) else str(data.get("text/plain", ""))
        if text:
            output_cells += 1
            if re.search(r"\bEMP-\d{3,}\b", text):
                employee_id_hits += 1

print("Cells with saved output:", output_cells)
print("Output cells containing employee-style IDs:", employee_id_hits)

print("\n=== AUDIT COMPLETE ===")

=== GIT WHITESPACE CHECK ===
PASS

=== SUMMARY RECONCILIATION ===
{
  "pipeline": "PQ-01 multi-feed ETL",
  "feeds": [
    {
      "table": "fact_payroll",
      "rows_in": 7344,
      "rows_passed": 7340,
      "rows_quarantined": 4,
      "reconciliation_ok": true,
      "quarantine_reasons": {
        "DUPLICATE_EMPLOYEE_RUN": 2,
        "UNMATCHED_EMPLOYEE_KEY": 2
      }
    },
    {
      "table": "fact_anomaly",
      "rows_in": 405,
      "rows_passed": 405,
      "rows_quarantined": 0,
      "reconciliation_ok": true,
      "quarantine_reasons": {},
      "warnings": [
        "No documented reason_code allowlist was found; values were not quarantined by reason code.",
        "scored_at is timezone-naive; UTC cannot be confirmed from the source schema.",
        "model_version naming convention is not fully specified.",
        "Attendance anomaly record IDs were validated through employee and month mapping rather than direct ID equality."
      ]
    },
    {
      "table": 

In [181]:
from pathlib import Path
import subprocess

root = Path.cwd()
gitignore = root / ".gitignore"

rule = "PQ-01_ETL_Validation_backup_*.ipynb"
content = gitignore.read_text(encoding="utf-8")

if rule not in content:
    gitignore.write_text(
        content.rstrip() + "\n\n# Local notebook backups\n" + rule + "\n",
        encoding="utf-8"
    )
    print("Added local notebook backup exclusion.")
else:
    print("Backup exclusion already exists.")

subprocess.run(
    ["git", "add", "--", ".gitignore"],
    cwd=root,
    check=True
)

check = subprocess.run(
    ["git", "check-ignore", "-v",
     "PQ-01_ETL_Validation_backup_20261020_093305.ipynb"],
    cwd=root,
    capture_output=True,
    text=True
)

print("\nBackup ignore verification:")
print(check.stdout.strip() if check.returncode == 0 else "FAILED")

Added local notebook backup exclusion.

Backup ignore verification:
.gitignore:33:PQ-01_ETL_Validation_backup_*.ipynb	PQ-01_ETL_Validation_backup_20261020_093305.ipynb


In [189]:
import subprocess
from pathlib import Path

root = Path.cwd()

git_name = "Your Name"
git_email = "your-email@example.com"

subprocess.run(
    ["git", "config", "--local", "user.name", git_name],
    cwd=root,
    check=True
)

subprocess.run(
    ["git", "config", "--local", "user.email", git_email],
    cwd=root,
    check=True
)

print("Git identity configured for this repository.")
print("Name:", subprocess.run(
    ["git", "config", "--local", "user.name"],
    cwd=root, capture_output=True, text=True, check=True
).stdout.strip())

print("Email:", subprocess.run(
    ["git", "config", "--local", "user.email"],
    cwd=root, capture_output=True, text=True, check=True
).stdout.strip())

Git identity configured for this repository.
Name: Your Name
Email: your-email@example.com


In [185]:
import subprocess
from pathlib import Path

root = Path.cwd()

git_name = input("Enter your Git author name: ").strip()
git_email = input("Enter your Git author email: ").strip()

if not git_name or not git_email or git_email == "your-email@example.com":
    raise ValueError("Enter a real author name and email before continuing.")

subprocess.run(
    ["git", "config", "--local", "user.name", git_name],
    cwd=root,
    check=True
)
subprocess.run(
    ["git", "config", "--local", "user.email", git_email],
    cwd=root,
    check=True
)

print("Git identity updated for this repository.")
print("Name:", subprocess.run(
    ["git", "config", "--local", "user.name"],
    cwd=root, capture_output=True, text=True, check=True
).stdout.strip())
print("Email:", subprocess.run(
    ["git", "config", "--local", "user.email"],
    cwd=root, capture_output=True, text=True, check=True
).stdout.strip())

Enter your Git author name:  Raathesh Kumar T
Enter your Git author email:  raatheshk@gmail.com


Git identity updated for this repository.
Name: Raathesh Kumar T
Email: raatheshk@gmail.com
